# Leakage-controlled brain tumor MRI classification: full experiment notebook
This is the notebook that produced the results in this repository. It was run on Kaggle (2 x Tesla T4, PyTorch 2.10, CUDA 12.8).

Inputs: the Sartaj Kaggle dataset "Brain Tumor Classification (MRI)" and the original Figshare brain tumor dataset of Cheng et al. (downloaded by the notebook). The external-screening cells at the end also need BDNeuro-MRI and PMRAM attached as Kaggle inputs.

Order: install, write `train_seed.py`, Figshare patient IDs and masks, splits, training on both GPUs, analysis, controls, explainability, timing, external screening, and pixel-level verification of hash matches. Finished training jobs are saved and skipped, so cells can be rerun safely.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q imagehash grad-cam fvcore h5py

In [ ]:
%%writefile train_seed.py
# train_seed.py : trains the paper's models (and baselines) for ONE seed on ONE GPU,
# selects epochs on the validation set only, and saves all predictions.
import os, sys, glob, copy, json, time, random, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split

SPLIT_SEED = 42                      # same split for every run, so the test set never changes
CLASSES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
NUM_CLASSES, NUM_EPOCHS, BATCH = 4, 12, 16
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)), transforms.CenterCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5), transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
test_transform = transforms.Compose([
    transforms.Resize((256, 256)), transforms.CenterCrop((224, 224)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class BrainTumorDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths, self.labels, self.transform = list(file_paths), list(labels), transform
    def __len__(self): return len(self.file_paths)
    def __getitem__(self, i):
        img = Image.open(self.file_paths[i]).convert("RGB")
        return (self.transform(img) if self.transform else img), self.labels[i]

class OptimizedResNet101(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        base = models.resnet101(weights=models.ResNet101_Weights.DEFAULT if pretrained else None)
        self.features = nn.Sequential(*list(base.children())[:-2])
        self.gap = nn.AdaptiveAvgPool2d((1, 1)); self.bn = nn.BatchNorm1d(2048)
        self.fc1 = nn.Linear(2048, 128); self.relu = nn.ReLU(); self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, 64); self.drop2 = nn.Dropout(0.2); self.out = nn.Linear(64, NUM_CLASSES)
    def forward(self, x):
        x = torch.flatten(self.gap(self.features(x)), 1); x = self.bn(x)
        x = self.drop1(self.relu(self.fc1(x))); x = self.drop2(self.relu(self.fc2(x)))
        return self.out(x)

def make_swin(pretrained=True):
    m = models.swin_v2_b(weights=models.Swin_V2_B_Weights.DEFAULT if pretrained else None)
    m.head = nn.Linear(m.head.in_features, NUM_CLASSES); return m

def make_convnext(pretrained=True):
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES); return m

def make_resnet50(pretrained=True):
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES); return m

def make_effb0(pretrained=True):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES); return m

# same optimiser, schedule, epochs and selection rule for every model; only the backbone changes
BUILDERS = {"resnet": (lambda: OptimizedResNet101(), 1e-4), "swin": (lambda: make_swin(), 5e-5),
            "convnext": (lambda: make_convnext(), 1e-4), "resnet50": (lambda: make_resnet50(), 1e-4),
            "effb0": (lambda: make_effb0(), 1e-4)}

def find_data_dir():
    for d in ["/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri",
              "/kaggle/input/brain-tumor-classification-mri"]:
        if os.path.exists(d): return d
    raise FileNotFoundError("Attach the Sartaj 'Brain Tumor Classification (MRI)' dataset")

def get_splits(split_file=None):
    split_file = split_file or os.environ.get("SPLIT_FILE", "/kaggle/working/split_grouped.json")
    """Reads the duplicate-grouped split made once by the launcher (same for every seed)."""
    sp = json.load(open(split_file))
    return ((sp["train_paths"], sp["train_y"]), (sp["val_paths"], sp["val_y"]), (sp["test_paths"], sp["test_y"]))

def all_images():
    d = find_data_dir(); paths, labels = [], []
    for phase in ["Training", "Testing"]:
        for c, cls in enumerate(CLASSES):
            for f in sorted(glob.glob(os.path.join(d, phase, cls, "*.jpg"))):
                paths.append(f); labels.append(c)
    return paths, labels

def make_grouped_split(out_file, thresh=5, seed=SPLIT_SEED):
    """Groups images whose perceptual hashes differ by <= thresh bits, then splits by GROUP
    (stratified) so no duplicate or near-duplicate can sit on both sides of a split."""
    import imagehash
    from sklearn.model_selection import StratifiedGroupKFold
    paths, labels = all_images(); n = len(paths)
    h = np.array([int(str(imagehash.phash(Image.open(p).convert("L"))), 16) for p in paths], dtype=np.uint64)
    parent = list(range(n))
    def find(a):
        while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for i0 in range(0, n, 256):
        blk = h[i0:i0 + 256]
        dist = np.unpackbits((blk[:, None] ^ h[None, :]).view(np.uint8).reshape(len(blk), n, 8), axis=2).sum(2)
        for a, b in zip(*np.where(dist <= thresh)):
            a += i0
            if b > a:
                ra, rb = find(a), find(b)
                if ra != rb: parent[rb] = ra
    groups = np.array([find(i) for i in range(n)]); y = np.array(labels)
    conflict = [g for g in np.unique(groups) if len(set(y[groups == g])) > 1]
    idx = np.arange(n)
    te_f = next(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(idx, y, groups))
    rest, test = te_f
    va_f = next(StratifiedGroupKFold(8, shuffle=True, random_state=seed).split(rest, y[rest], groups[rest]))
    train, val = rest[va_f[0]], rest[va_f[1]]
    # verify: no group appears in more than one split
    gs = [set(groups[train]), set(groups[val]), set(groups[test])]
    assert not (gs[0] & gs[1]) and not (gs[0] & gs[2]) and not (gs[1] & gs[2])
    sp = {k + "_paths": [paths[i] for i in v] for k, v in [("train", train), ("val", val), ("test", test)]}
    sp.update({k + "_y": [int(y[i]) for i in v] for k, v in [("train", train), ("val", val), ("test", test)]})
    multi = [g for g in np.unique(groups) if (groups == g).sum() > 1]
    sp["dup_groups"] = [[[paths[i], int(y[i])] for i in np.where(groups == g)[0]] for g in multi]
    sizes = np.bincount(groups)[np.unique(groups)]
    sp["info"] = {"images": n, "groups": int(len(sizes)), "images_in_duplicate_groups": int(sizes[sizes > 1].sum()),
                  "largest_group": int(sizes.max()), "groups_with_conflicting_labels": len(conflict),
                  "hash_threshold": thresh, "counts": {k: np.bincount(y[v], minlength=NUM_CLASSES).tolist()
                                                       for k, v in [("train", train), ("val", val), ("test", test)]}}
    json.dump(sp, open(out_file, "w"), indent=1)
    return sp["info"]

def make_patient_split(out_file, map_csv, thresh=5, seed=SPLIT_SEED):
    """Patient-aware split. Images are linked if their perceptual hashes differ by <= thresh bits OR they share a
    Figshare patient ID; linked images form one group, and whole groups are split (stratified 70/10/20).
    Images whose Sartaj label disagrees with the original Figshare label are left out and listed."""
    import imagehash, pandas as pd
    from sklearn.model_selection import StratifiedGroupKFold
    tail = lambda p: "/".join(str(p).split("/")[-3:])
    mp = pd.read_csv(map_csv, dtype={"pid": str})
    mp = {tail(r.path): (None if pd.isna(r.pid) else r.pid, None if pd.isna(r.fig_label) else int(r.fig_label)) for r in mp.itertuples()}
    paths_all, labels_all = all_images()
    pid_all = [mp.get(tail(p), (None, None))[0] for p in paths_all]
    fl_all = [mp.get(tail(p), (None, None))[1] for p in paths_all]
    excluded = [{"path": p, "sartaj_label": int(y), "figshare_label": fl, "pid": pd_}
                for p, y, fl, pd_ in zip(paths_all, labels_all, fl_all, pid_all) if fl is not None and fl != y]
    keep = [i for i in range(len(paths_all)) if not (fl_all[i] is not None and fl_all[i] != labels_all[i])]
    paths = [paths_all[i] for i in keep]; y = np.array([labels_all[i] for i in keep]); pid = [pid_all[i] for i in keep]; n = len(paths)
    h = np.array([int(str(imagehash.phash(Image.open(p).convert("L"))), 16) for p in paths], dtype=np.uint64)
    parent = list(range(n))
    def find(a):
        while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    for i0 in range(0, n, 256):                                   # link near-duplicates
        blk = h[i0:i0 + 256]
        dist = np.unpackbits((blk[:, None] ^ h[None, :]).view(np.uint8).reshape(len(blk), n, 8), axis=2).sum(2)
        for a, b in zip(*np.where(dist <= thresh)):
            if b > a + i0: union(a + i0, b)
    first = {}                                                    # link images of the same patient
    for i, p_ in enumerate(pid):
        if p_ is None: continue
        if p_ in first: union(first[p_], i)
        else: first[p_] = i
    groups = np.array([find(i) for i in range(n)]); idx = np.arange(n)
    rest, test = next(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(idx, y, groups))
    va_f = next(StratifiedGroupKFold(8, shuffle=True, random_state=seed).split(rest, y[rest], groups[rest]))
    train, val = rest[va_f[0]], rest[va_f[1]]
    gs = [set(groups[train]), set(groups[val]), set(groups[test])]
    assert not (gs[0] & gs[1]) and not (gs[0] & gs[2]) and not (gs[1] & gs[2]), "a group crosses splits"
    ps = [{pid[i] for i in v if pid[i] is not None} for v in (train, val, test)]
    assert not (ps[0] & ps[2]) and not (ps[1] & ps[2]) and not (ps[0] & ps[1]), "a patient crosses splits"
    sp = {}
    for k, v in [("train", train), ("val", val), ("test", test)]:
        sp[k + "_paths"] = [paths[i] for i in v]; sp[k + "_y"] = [int(y[i]) for i in v]; sp[k + "_pid"] = [pid[i] for i in v]
    sizes = np.bincount(groups)[np.unique(groups)]
    sp["excluded_label_conflicts"] = excluded
    sp["info"] = {"images_total": len(paths_all), "excluded_label_conflicts": len(excluded), "images_used": n,
                  "groups": int(len(sizes)), "largest_group": int(sizes.max()), "hash_threshold": thresh,
                  "patients": {k: len(s_) for k, s_ in zip(["train", "val", "test"], ps)},
                  "patients_crossing_splits": 0,
                  "counts": {k: np.bincount(y[v], minlength=NUM_CLASSES).tolist() for k, v in [("train", train), ("val", val), ("test", test)]},
                  "tumor_images_with_patient_id": {k: int(sum(pid[i] is not None for i in v if y[i] != 2)) for k, v in [("train", train), ("val", val), ("test", test)]},
                  "tumor_images_without_patient_id": {k: int(sum(pid[i] is None for i in v if y[i] != 2)) for k, v in [("train", train), ("val", val), ("test", test)]}}
    json.dump(sp, open(out_file, "w"), indent=1)
    return sp["info"]

def set_seed(s):
    os.environ["PYTHONHASHSEED"] = str(s); random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def train(model, opt, sch, train_loader, val_loader, name, device, log):
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_acc, best_ep, best_wts = -1.0, 0, None
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for ep in range(NUM_EPOCHS):
        model.train(); tl = tc = tn = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device); opt.zero_grad()
            out = model(x); loss = crit(out, y); loss.backward(); opt.step()
            tl += loss.item() * x.size(0); tc += (out.argmax(1) == y).sum().item(); tn += x.size(0)
        sch.step(); model.eval(); vl = vc = vn = 0
        with torch.no_grad():
            for x, y in val_loader:                       # VALIDATION set, never test
                x, y = x.to(device), y.to(device); out = model(x)
                vl += crit(out, y).item() * x.size(0); vc += (out.argmax(1) == y).sum().item(); vn += x.size(0)
        for k, v in zip(hist, [tl/tn, tc/tn, vl/vn, vc/vn]): hist[k].append(v)
        log(f"{name} epoch {ep+1}/{NUM_EPOCHS} train_acc {tc/tn:.4f} val_acc {vc/vn:.4f} val_loss {vl/vn:.4f}")
        if vc / vn > best_acc:
            best_acc, best_ep, best_wts = vc / vn, ep + 1, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_wts)
    log(f"{name}: kept epoch {best_ep} (val_acc {best_acc:.4f})")
    return model, hist, best_ep, best_acc

@torch.no_grad()
def predict_views(model, loader, device):
    model.eval(); o, f = [], []
    for x, _ in loader:
        x = x.to(device)
        o.append(F.softmax(model(x), 1).cpu()); f.append(F.softmax(model(torch.flip(x, [3])), 1).cpu())
    return torch.cat(o).numpy(), torch.cat(f).numpy()

def run(seed, out_dir, model_names):
    os.makedirs(out_dir, exist_ok=True)
    logf = open(os.path.join(out_dir, f"log_seed{seed}.txt"), "a")
    def log(m):
        line = f"[seed {seed} | {time.strftime('%H:%M:%S')}] {m}"; print(line, flush=True); logf.write(line + "\n"); logf.flush()
    todo = [n for n in model_names if not (os.path.exists(os.path.join(out_dir, f"probs_{n}_seed{seed}.npz"))
                                           and os.path.exists(os.path.join(out_dir, f"{n}_seed{seed}.pt")))]
    if not todo: log("all requested models already done, skipping"); return
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log(f"device {device} {torch.cuda.get_device_name(0) if device.type=='cuda' else ''}, torch {torch.__version__}, training {todo}")
    (tr_p, tr_y), (va_p, va_y), (te_p, te_y) = get_splits()
    log(f"split train {len(tr_p)} | val {len(va_p)} | test {len(te_p)}")
    def wi(w): np.random.seed(seed + w); random.seed(seed + w)
    set_seed(seed); g = torch.Generator().manual_seed(seed)          # shared generator: same order as the original run
    tl = DataLoader(BrainTumorDataset(tr_p, tr_y, train_transform), batch_size=BATCH, shuffle=True, num_workers=2, worker_init_fn=wi, generator=g)
    vl = DataLoader(BrainTumorDataset(va_p, va_y, test_transform), batch_size=BATCH, shuffle=False, num_workers=2)
    te = DataLoader(BrainTumorDataset(te_p, te_y, test_transform), batch_size=BATCH, shuffle=False, num_workers=2)
    meta_path = os.path.join(out_dir, f"meta_seed{seed}.json")
    meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {}
    for name in [n for n in ["resnet", "swin", "convnext", "resnet50", "effb0"] if n in todo]:
        build, lr = BUILDERS[name]
        loader = tl
        if name not in ("resnet", "swin"):                             # own generator, independent of the paper's two branches
            loader = DataLoader(BrainTumorDataset(tr_p, tr_y, train_transform), batch_size=BATCH, shuffle=True, num_workers=2,
                                worker_init_fn=wi, generator=torch.Generator().manual_seed(seed))
        set_seed(seed); m = build().to(device)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4); sch = CosineAnnealingLR(opt, T_max=NUM_EPOCHS)
        t0 = time.time(); m, h, ep, acc = train(m, opt, sch, loader, vl, name, device, log)
        v_o, v_f = predict_views(m, vl, device); t_o, t_f = predict_views(m, te, device)
        np.savez(os.path.join(out_dir, f"probs_{name}_seed{seed}.npz"), val_y=np.array(va_y), test_y=np.array(te_y),
                 val_paths=np.array(va_p), test_paths=np.array(te_p), train_paths=np.array(tr_p),
                 val_orig=v_o, val_flip=v_f, test_orig=t_o, test_flip=t_f)
        torch.save(m.state_dict(), os.path.join(out_dir, f"{name}_seed{seed}.pt"))
        meta[name] = {"hist": h, "best_epoch": ep, "best_val_acc": acc, "train_minutes": (time.time() - t0) / 60}
        json.dump(meta, open(meta_path, "w"), indent=1)
        del m; torch.cuda.empty_cache()
    combine(seed, out_dir); log("DONE")

def combine(seed, out_dir):
    """write the combined ResNet+Swin file used by the analysis, if both branches are available"""
    f = os.path.join(out_dir, f"preds_seed{seed}.npz")
    pr, ps = [os.path.join(out_dir, f"probs_{n}_seed{seed}.npz") for n in ["resnet", "swin"]]
    if os.path.exists(f) or not (os.path.exists(pr) and os.path.exists(ps)): return
    a, b = np.load(pr, allow_pickle=True), np.load(ps, allow_pickle=True)
    np.savez(f, val_y=a["val_y"], test_y=a["test_y"], val_paths=a["val_paths"], test_paths=a["test_paths"], train_paths=a["train_paths"],
             **{f"resnet_{k}": a[k] for k in ["val_orig", "val_flip", "test_orig", "test_flip"]},
             **{f"swin_{k}": b[k] for k in ["val_orig", "val_flip", "test_orig", "test_flip"]})

if __name__ == "__main__":
    import traceback
    ap = argparse.ArgumentParser()
    ap.add_argument("--jobs", help="JSON list of {split, out, models, seed}")
    ap.add_argument("--seeds", type=int, nargs="+"); ap.add_argument("--models", nargs="+", default=["resnet", "swin"])
    ap.add_argument("--out", default="/kaggle/working/runs_grouped"); a = ap.parse_args()
    jobs = json.load(open(a.jobs)) if a.jobs else [{"split": None, "out": a.out, "models": a.models, "seed": s} for s in a.seeds]
    for j in jobs:
        if j.get("split"): os.environ["SPLIT_FILE"] = j["split"]
        try:
            run(j["seed"], j["out"], j["models"])
        except Exception as e:
            print(f"[seed {j['seed']} {j['models']} {j['out']}] FAILED: {type(e).__name__}: {e}", flush=True)
            traceback.print_exc(); torch.cuda.empty_cache()

In [ ]:
# ===== Original Figshare data: patient IDs (M2) and tumor masks (M12). Reuses files already in /kaggle/working =====
import os, glob, zipfile, shutil, subprocess, urllib.request, numpy as np, pandas as pd, imagehash, importlib, train_seed
from PIL import Image
importlib.reload(train_seed)
W = "/kaggle/working"; FIG = f"{W}/figshare"; JPG = f"{FIG}/jpg"; MSK = f"{FIG}/mask"; RAW = f"{FIG}/raw"
for d_ in (JPG, MSK, RAW): os.makedirs(d_, exist_ok=True)
NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
MAP = {1: 1, 2: 0, 3: 3}                                    # Figshare labels: 1 meningioma, 2 glioma, 3 pituitary

# 1. find the original .mat files: attached, uploaded zip, or download
def find_mats():
    return sorted({m for m in glob.glob("/kaggle/input/**/*.mat", recursive=True) + glob.glob(f"{FIG}/**/*.mat", recursive=True)
                   if "cvind" not in os.path.basename(m).lower()})
def unpack(zip_path):
    """unpack a Figshare archive and the four inner brainTumorDataPublic zips"""
    try:
        with zipfile.ZipFile(zip_path) as z: z.extractall(RAW)
    except zipfile.BadZipFile:
        print("  not a valid zip:", os.path.basename(zip_path)); return
    inner = sorted(glob.glob(f"{RAW}/**/*.zip", recursive=True))
    bad = []
    for f in inner:
        try:
            with zipfile.ZipFile(f) as z: z.extractall(f"{RAW}/mat")
        except zipfile.BadZipFile: bad.append(f)
    if bad:                                                  # some copies are one archive split in parts: join and repair
        joined = f"{RAW}/joined.zip"
        with open(joined, "wb") as out:
            for f in bad:
                with open(f, "rb") as src: shutil.copyfileobj(src, out)
        subprocess.run(["zip", "-FF", joined, "--out", f"{RAW}/fixed.zip"], input=b"y\n", capture_output=True)
        try:
            with zipfile.ZipFile(f"{RAW}/fixed.zip") as z: z.extractall(f"{RAW}/mat")
        except Exception as e: print("  could not repair split archive:", e)

mats = find_mats()
if len(mats) < 3000:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):            # a zip you uploaded yourself
        print("Unpacking uploaded archive:", z); unpack(z)
    mats = find_mats()
if len(mats) < 3000:
    zp = f"{FIG}/figshare.zip"
    if not (os.path.exists(zp) and zipfile.is_zipfile(zp)):
        for url in ["https://ndownloader.figshare.com/articles/1512427/versions/5",
                    "https://figshare.com/ndownloader/articles/1512427/versions/5"]:
            print("Trying download:", url)
            subprocess.run(["wget", "-q", "-O", zp, "--user-agent=Mozilla/5.0", url])
            if os.path.exists(zp) and zipfile.is_zipfile(zp): break
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) Chrome/124 Safari/537.36"})
                with urllib.request.urlopen(req, timeout=600) as r, open(zp, "wb") as f: shutil.copyfileobj(r, f)
            except Exception as e: print("  failed:", e)
            if os.path.exists(zp) and zipfile.is_zipfile(zp): break
    if os.path.exists(zp) and zipfile.is_zipfile(zp): unpack(zp)
    mats = find_mats()

# 2. build the external image list (with patient IDs if the .mat files are available)
idx_csv = f"{FIG}/figshare_index.csv"
if len(mats) >= 3000:
    PATIENT_IDS = True
    def read_mat(f):
        try:
            import h5py
            with h5py.File(f, "r") as h:
                cj = h["cjdata"]; img = np.array(cj["image"]).T; lab = int(np.array(cj["label"]).ravel()[0])
                pid = "".join(chr(int(c)) for c in np.array(cj["PID"]).ravel()); msk = np.array(cj["tumorMask"]).T
        except OSError:                                     # older MATLAB format
            from scipy.io import loadmat
            cj = loadmat(f)["cjdata"][0, 0]; img = cj["image"]; lab = int(cj["label"].ravel()[0]); pid = str(cj["PID"].ravel()[0]); msk = cj["tumorMask"]
        img = img.astype(np.float32); img = (img - img.min()) / max(img.max() - img.min(), 1e-6) * 255
        return img.astype(np.uint8), MAP[lab], pid.strip(), (np.asarray(msk) > 0).astype(np.uint8) * 255
    if not os.path.exists(idx_csv):
        rec = []
        for f in sorted(mats, key=lambda m: int("".join(ch for ch in os.path.basename(m) if ch.isdigit()) or 0)):
            img, lab, pid, msk = read_mat(f); stem = os.path.splitext(os.path.basename(f))[0]
            out, mout = f"{JPG}/{stem}.png", f"{MSK}/{stem}.png"
            Image.fromarray(img).save(out); Image.fromarray(msk).save(mout)
            rec.append({"path": out, "mask": mout, "label": lab, "pid": pid})
        pd.DataFrame(rec).to_csv(idx_csv, index=False)
else:
    # fallback: PNG copy of Figshare (folders 1, 2, 3). No patient IDs, so the test is image-level only.
    pngs = [p for p in glob.glob("/kaggle/input/**/*.png", recursive=True) if os.path.basename(os.path.dirname(p)) in ("1", "2", "3")]
    assert len(pngs) >= 3000, ("Figshare data not found. Attach it (original .mat files, your own uploaded zip, "
                               "or the PNG copy with folders 1, 2, 3) and run this cell again.")
    PATIENT_IDS = False
    print("WARNING: using the PNG copy of Figshare. It has no patient IDs, so this external test is IMAGE-LEVEL only.")
    if not os.path.exists(idx_csv):
        rec = []
        for p in sorted(pngs):
            im = Image.open(p)
            a = np.array(im).astype(np.float32)
            if a.ndim == 3: a = a[..., :3].mean(2)
            a = (a - a.min()) / max(a.max() - a.min(), 1e-6) * 255
            out = f"{JPG}/png_{os.path.basename(os.path.dirname(p))}_{os.path.basename(p)}"
            Image.fromarray(a.astype(np.uint8)).save(out)
            rec.append({"path": out, "label": MAP[int(os.path.basename(os.path.dirname(p)))], "pid": "img_" + os.path.basename(p)})
        pd.DataFrame(rec).to_csv(idx_csv, index=False)
ext = pd.read_csv(idx_csv, dtype={"pid": str})
print(f"External images: {len(ext)} from {ext.pid.nunique()} {'patients' if PATIENT_IDS else 'images (no patient IDs)'} | per class:",
      {NAMES[k]: int(v) for k, v in ext.label.value_counts().sort_index().items()})

assert PATIENT_IDS, "The original Figshare .mat files are needed for patient IDs (the PNG copy has none)."

In [ ]:
# ===== Map every Sartaj image to a Figshare patient ID (needs the original Figshare .mat files) =====
import os, numpy as np, pandas as pd, imagehash, importlib, train_seed
from PIL import Image
importlib.reload(train_seed); from train_seed import all_images
W = "/kaggle/working"; NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
if os.path.exists(f"{W}/sartaj_all_to_figshare_pid.csv"):
    print("Patient mapping already exists, reusing it."); mp = pd.read_csv(f"{W}/sartaj_all_to_figshare_pid.csv", dtype={"pid": str})
else:
  fig = pd.read_csv(f"{W}/figshare/figshare_index.csv", dtype={"pid": str})
  assert "mask" in fig.columns, "Old Figshare index without masks: delete /kaggle/working/figshare/figshare_index.csv and run the previous cell again."
  assert not fig.pid.str.startswith("img_").any(), "Patient IDs are needed: use the original Figshare .mat files, not the PNG copy."
  paths, labels = all_images()
  H = lambda ps: np.array([int(str(imagehash.phash(Image.open(p).convert("L"))), 16) for p in ps], dtype=np.uint64)
  hs, hf = H(paths), H(fig.path.tolist())
  rows = []
  for i in range(0, len(hs), 256):
      x = (hs[i:i+256, None] ^ hf[None, :]).view(np.uint8).reshape(len(hs[i:i+256]), len(hf), 8)
      d = np.unpackbits(x, axis=2).sum(2); j = d.argmin(1); m = d.min(1)
      for k in range(len(j)):
          ok = m[k] <= 5
          rows.append({"path": paths[i+k], "label": labels[i+k], "fig_dist": int(m[k]),
                       "pid": fig.pid[j[k]] if ok else None, "fig_label": int(fig.label[j[k]]) if ok else None,
                       "fig_path": fig.path[j[k]] if ok else None, "fig_mask": fig["mask"][j[k]] if ok else None})
  mp = pd.DataFrame(rows); mp.to_csv(f"{W}/sartaj_all_to_figshare_pid.csv", index=False)
print("Share of Sartaj images matched to a Figshare patient, by class:",
      {NAMES[c]: round(float(mp[mp.label == c].pid.notna().mean()), 3) for c in range(4)})
print(f"Distinct Figshare patients found in Sartaj: {mp.pid.nunique()}")
mt = mp[mp.pid.notna()]
ct = pd.crosstab(mt.label.map(dict(enumerate(NAMES))), mt.fig_label.map(dict(enumerate(NAMES))), rownames=["Sartaj label"], colnames=["Figshare label"])
ct.to_csv(f"{W}/table_label_agreement_all.csv")
print(f"\nLabel agreement on {len(mt)} matched images: {100*(mt.label == mt.fig_label).mean():.2f}%\n", ct.to_string())

In [ ]:
# ===== Splits (M2, M3, M10): hash thresholds 0/3/5/8, patient-aware split, label-conflicting links, image table =====
import os, json, importlib, numpy as np, pandas as pd, imagehash, matplotlib.pyplot as plt, train_seed
from PIL import Image
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
importlib.reload(train_seed); from train_seed import all_images, make_grouped_split, make_patient_split
W = "/kaggle/working"; NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]; THRESHOLDS = [0, 3, 5, 8]
tail = lambda p: "/".join(str(p).split("/")[-3:])
paths, labels = all_images(); y = np.array(labels); n = len(paths)
if os.path.exists(f"{W}/phash_all.npy"): h = np.load(f"{W}/phash_all.npy")
else:
    h = np.array([int(str(imagehash.phash(Image.open(p).convert("L"))), 16) for p in paths], dtype=np.uint64); np.save(f"{W}/phash_all.npy", h)
D = np.zeros((n, n), np.uint8)                                   # all pairwise Hamming distances (3,264 x 3,264)
for i in range(0, n, 256):
    D[i:i+256] = np.unpackbits((h[i:i+256, None] ^ h[None, :]).view(np.uint8).reshape(len(h[i:i+256]), n, 8), axis=2).sum(2)
iu, ju = np.triu_indices(n, 1); dpair = D[iu, ju]
def components(adj_pairs):
    a, b = adj_pairs; g = coo_matrix((np.ones(len(a)), (a, b)), shape=(n, n))
    return connected_components(g, directed=False)[1]

# 1. grouped splits at each threshold (training uses train_seed.make_grouped_split, the same code as the paper)
rows, groups_at = [], {}
for t in THRESHOLDS:
    f = f"{W}/split_grouped_t{t}.json"
    if not os.path.exists(f): make_grouped_split(f, thresh=t)
    sp = json.load(open(f)); m = dpair <= t; g = components((iu[m], ju[m])); groups_at[t] = g
    sizes = np.bincount(g); multi = sizes[sizes > 1]
    cross = m & (y[iu] != y[ju])
    conflict_groups = sum(len(set(y[g == k])) > 1 for k in np.where(sizes > 1)[0])
    trs, tes = set(sp["train_paths"]) | set(sp["val_paths"]), set(sp["test_paths"])
    tr = np.array([p in trs for p in paths]); te = np.array([p in tes for p in paths])
    dmin = D[np.ix_(te, tr)].min(1)
    rows.append({"Hash threshold (bits)": t, "Linked pairs": int(m.sum()), "Label-conflicting pairs": int(cross.sum()),
                 "Groups with >1 image": int(len(multi)), "Images in those groups": int(multi.sum()), "Largest group": int(sizes.max()),
                 "Groups with mixed labels": int(conflict_groups), "Train / val / test": f"{len(sp['train_paths'])} / {len(sp['val_paths'])} / {len(sp['test_paths'])}",
                 "Test images with a train/val image within 5 bits": int((dmin <= 5).sum()),
                 "within 8 bits": int((dmin <= 8).sum()), "within 12 bits": int((dmin <= 12).sum())})
TH = pd.DataFrame(rows); TH.to_csv(f"{W}/table_threshold_grouping.csv", index=False); print(TH.to_string(index=False))

# 2. label-conflicting links at threshold 5: how many are direct edges, and what they look like
m5 = (dpair <= 5) & (y[iu] != y[ju])
cl = pd.DataFrame({"image_a": [tail(paths[i]) for i in iu[m5]], "label_a": [NAMES[y[i]] for i in iu[m5]],
                   "image_b": [tail(paths[j]) for j in ju[m5]], "label_b": [NAMES[y[j]] for j in ju[m5]], "hamming": dpair[m5]})
cl.sort_values("hamming").to_csv(f"{W}/table_label_conflicting_links_t5.csv", index=False)
print(f"\nThreshold 5: {m5.sum()} direct links join images with different labels (distances: {np.bincount(dpair[m5], minlength=6).tolist()} for 0..5 bits).")
k = min(6, len(cl))
if k:
    fig, ax = plt.subplots(k, 2, figsize=(6, 3 * k), squeeze=False)
    for r, (a, b) in enumerate(zip(iu[m5][:k], ju[m5][:k])):
        for c, i in enumerate([a, b]):
            ax[r, c].imshow(Image.open(paths[i]).convert("L"), cmap="gray"); ax[r, c].axis("off")
            ax[r, c].set_title(f"{NAMES[y[i]]} | {tail(paths[i])}\nHamming {D[a, b]}", fontsize=7)
    plt.tight_layout(); plt.savefig(f"{W}/Fig_label_conflicting_links_t5.png", dpi=200); plt.show()

# 3. patient-aware split (near-duplicates OR same Figshare patient -> same group); Figshare label conflicts excluded
PSPLIT = f"{W}/split_patient.json"
if not os.path.exists(PSPLIT): make_patient_split(PSPLIT, f"{W}/sartaj_all_to_figshare_pid.csv", thresh=5)
sp = json.load(open(PSPLIT)); print("\nPatient-aware split:", json.dumps(sp["info"], indent=1))
trv = {p for k_ in ["train", "val"] for p in sp[k_ + "_pid"] if p}; tst = {p for p in sp["test_pid"] if p}
print(f"Patient-aware split: test patients {len(tst)}, also in train/val: {len(tst & trv)} (must be 0)")

# 4. how much patient overlap the near-duplicate split (threshold 5, used in the current paper) still had (M2)
mp = pd.read_csv(f"{W}/sartaj_all_to_figshare_pid.csv", dtype={"pid": str}); pid_of = {tail(r.path): r.pid for r in mp.itertuples() if isinstance(r.pid, str)}
g5 = json.load(open(f"{W}/split_grouped_t5.json"))
tr5 = {pid_of.get(tail(p)) for k_ in ["train_paths", "val_paths"] for p in g5[k_]} - {None}
te5 = [pid_of.get(tail(p)) for p in g5["test_paths"]]
te5_p = {p for p in te5 if p}; leaked_imgs = sum(1 for p in te5 if p and p in tr5)
print(f"Near-duplicate split (t=5): {len(te5_p)} test patients identified, {len(te5_p & tr5)} of them also in train/val; "
      f"{leaked_imgs} of {sum(p is not None for p in te5)} identified tumor test images come from such patients.")
json.dump({"t5_test_patients": len(te5_p), "t5_test_patients_also_in_trainval": len(te5_p & tr5), "t5_test_images_from_leaked_patients": leaked_imgs,
           "t5_test_tumor_images_with_pid": int(sum(p is not None for p in te5))}, open(f"{W}/patient_overlap_in_grouped_split.json", "w"), indent=1)

# 5. image table for the reproducibility package (M10): file, label, folder, hash, group ids, patient id, split membership
mem = {}
for nm, f in [("split_grouped_t5", f"{W}/split_grouped_t5.json"), ("split_patient", PSPLIT)]:
    s_ = json.load(open(f)); mem[nm] = {tail(p): part for part in ["train", "val", "test"] for p in s_[part + "_paths"]}
it = pd.DataFrame({"file": [tail(p) for p in paths], "label": [NAMES[v] for v in y], "original_folder": [tail(p).split("/")[0] for p in paths],
                   "phash_hex": [f"{int(v):016x}" for v in h], **{f"group_t{t}": groups_at[t] for t in THRESHOLDS},
                   "figshare_patient_id": [pid_of.get(tail(p)) for p in paths]})
for nm in mem: it[nm] = it["file"].map(mem[nm]).fillna("excluded")
it.to_csv(f"{W}/image_table_hash_groups_splits.csv", index=False); print(f"\nSaved image_table_hash_groups_splits.csv ({len(it)} images)")

In [ ]:
# ===== Launch all training jobs on both GPUs (finished jobs are skipped, so this cell can be re-run safely) =====
import os, sys, json, subprocess
from torchvision import models
for f in [lambda: models.resnet101(weights=models.ResNet101_Weights.DEFAULT), lambda: models.swin_v2_b(weights=models.Swin_V2_B_Weights.DEFAULT),
          lambda: models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT), lambda: models.resnet50(weights=models.ResNet50_Weights.DEFAULT),
          lambda: models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)]: f()     # download weights once, before the GPU processes
W = "/kaggle/working"; SEEDS = [42, 43, 44, 45]
A = [{"split": f"{W}/split_patient.json", "out": f"{W}/runs_patient", "models": ["resnet", "swin"], "seed": s} for s in SEEDS]          # M2: main result
C = [{"split": f"{W}/split_grouped_t{t}.json", "out": f"{W}/runs_grouped_t{t}", "models": ["resnet", "swin"], "seed": 42} for t in [0, 3, 5, 8]]  # M3
B = [{"split": f"{W}/split_patient.json", "out": f"{W}/runs_patient_baselines", "models": ["convnext", "resnet50", "effb0"], "seed": s} for s in SEEDS]  # M9
def done(j):
    return all(os.path.exists(f"{j['out']}/probs_{m}_seed{j['seed']}.npz") for m in j["models"])
order = [j for j in A + C + B if not done(j)]                     # most important first: if time runs out, the main result is safe
q = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.split("\n")
free = [int(l.split(",")[0]) for l in q if l.strip() and int(l.split(",")[1]) < 1000]
assert free, "No free GPU: Run > Restart & clear cell outputs, then run from the top."
print("Free GPUs:", free, "| jobs to run:", len(order))
procs = {}
for k, g in enumerate(free):
    jobs = order[k::len(free)]
    if not jobs: continue
    jf = f"{W}/jobs_gpu{g}.json"; json.dump(jobs, open(jf, "w"), indent=1)
    procs[g] = subprocess.Popen([sys.executable, "train_seed.py", "--jobs", jf], env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)),
                                stdout=open(f"{W}/train_gpu{g}.out", "a"), stderr=subprocess.STDOUT)
    print(f"GPU {g}: {len(jobs)} jobs ->", [(os.path.basename(j['out']), j['seed']) for j in jobs])

In [ ]:
# ===== Monitor: last line from each GPU every 2 minutes. Stopping this cell does NOT stop training. =====
import time
while procs and any(p.poll() is None for p in procs.values()):
    for g in procs:
        L = open(f"{W}/train_gpu{g}.out").read().strip().splitlines(); print(f"GPU {g}: {L[-1] if L else 'starting...'}")
    print("-" * 90); time.sleep(120)
left = [j for j in A + C + B if not done(j)]
fails = [l for g in procs for l in open(f"{W}/train_gpu{g}.out") if "FAILED" in l]
print("All jobs finished." if not left else f"{len(left)} jobs not finished: run the launch cell again.", "\n".join(fails))

In [ ]:
# ===== ANALYSIS: primary model, baselines, paired statistics (M4, M7, M8, M9, M11) =====
import os, glob, re, json, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from math import sqrt
from scipy.stats import binomtest, beta
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.preprocessing import label_binarize
W = "/kaggle/working"; NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]; NT = 2
PRIMARY = "Equal fusion, no TTA"          # reviewer M8: the simpler no-TTA ensemble is the primary model
RNG = np.random.default_rng(0); NBOOT = 5000
PARAMS = {"ResNet101": 42.78, "Swin-V2-B": 86.91, "ConvNeXt-T": 27.82, "ResNet50": 23.52, "EfficientNet-B0": 4.01}
BASE = {"resnet50": "ResNet50", "effb0": "EfficientNet-B0", "convnext": "ConvNeXt-T"}

def wilson(k, n, z=1.96):
    p = k/n; d = 1 + z*z/n; c = (p + z*z/(2*n))/d; h = z*sqrt(p*(1-p)/n + z*z/(4*n*n))/d; return c - h, c + h
def exact_ci(k, n, a=0.05):                                   # Clopper-Pearson
    lo = 0.0 if k == 0 else beta.ppf(a/2, k, n-k+1); hi = 1.0 if k == n else beta.ppf(1-a/2, k+1, n-k); return lo, hi
def ece(p, y, bins=15):
    conf, ok = p.max(1), p.argmax(1) == y; e = 0.0; ed = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(ed[:-1], ed[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return e
def brier(p, y): return np.mean(np.sum((p - np.eye(p.shape[1])[y])**2, 1))
def mf1(p, y): return f1_score(y, p.argmax(1), average="macro")
def tune(rv, sv, y):
    ws = np.round(np.arange(0, 1.0001, .01), 2); sc = np.array([mf1(w*rv + (1-w)*sv, y) for w in ws])
    b = ws[sc == sc.max()]; return float(b[np.argmin(abs(b - .5))])
def boot_ci(stat, n, nboot=NBOOT):
    idx = RNG.integers(0, n, (nboot, n)); v = np.array([stat(i) for i in idx]); return np.percentile(v, [2.5, 97.5])

def load_run(run_dir, base_dir=None):
    seeds, C, Y, VAL = [], {}, {}, {}
    for f in sorted(glob.glob(f"{run_dir}/preds_seed*.npz")):
        s = int(re.findall(r"seed(\d+)", f)[0]); d = np.load(f, allow_pickle=True); seeds.append(s)
        R = {k: d[f"resnet_{k}"] for k in ["val_orig", "val_flip", "test_orig", "test_flip"]}
        S = {k: d[f"swin_{k}"] for k in ["val_orig", "val_flip", "test_orig", "test_flip"]}
        yv, yt = d["val_y"], d["test_y"]; rt, st = (R["test_orig"] + R["test_flip"])/2, (S["test_orig"] + S["test_flip"])/2
        w_no = tune(R["val_orig"], S["val_orig"], yv); w_t = tune((R["val_orig"] + R["val_flip"])/2, (S["val_orig"] + S["val_flip"])/2, yv)
        c = {"ResNet101, no TTA": R["test_orig"], "ResNet101, TTA": rt, "Swin-V2-B, no TTA": S["test_orig"], "Swin-V2-B, TTA": st,
             "Equal fusion, no TTA": (R["test_orig"] + S["test_orig"])/2, "Equal fusion, TTA": (rt + st)/2,
             "Fixed 46/54 fusion, no TTA": .46*R["test_orig"] + .54*S["test_orig"],
             "Tuned fusion, no TTA": w_no*R["test_orig"] + (1-w_no)*S["test_orig"], "Tuned fusion, TTA": w_t*rt + (1-w_t)*st}
        if base_dir:
            for m, nm in BASE.items():
                pf = f"{base_dir}/probs_{m}_seed{s}.npz"
                if os.path.exists(pf):
                    b = np.load(pf, allow_pickle=True); assert (b["test_y"] == yt).all()
                    c[f"{nm}, no TTA"] = b["test_orig"]; c[f"{nm}, TTA"] = (b["test_orig"] + b["test_flip"])/2
            if "ConvNeXt-T, no TTA" in c: c["ConvNeXt-T + Swin-V2-B fusion, no TTA"] = (c["ConvNeXt-T, no TTA"] + S["test_orig"])/2
        C[s], Y[s], VAL[s] = c, yt, {"w_tuned_noTTA": w_no, "w_tuned_TTA": w_t}
    assert seeds, f"no runs in {run_dir}"
    y0 = Y[seeds[0]]; assert all((Y[s] == y0).all() for s in seeds), "seeds must share one test set"
    # compute-matched comparisons: two seeds of the same backbone averaged (two networks, like the fusion)
    pairs = [(seeds[i], seeds[i+1]) for i in range(0, len(seeds) - 1, 2)]
    for a, b in pairs:
        for nm in ["Swin-V2-B", "ResNet101"]:
            C[a][f"2 x {nm} (two seeds averaged), no TTA"] = (C[a][f"{nm}, no TTA"] + C[b][f"{nm}, no TTA"])/2
    return seeds, C, y0, VAL, pairs

def analyse(run_dir, base_dir, tag):
    print(f"\n{'='*30} {tag} {'='*30}")
    seeds, C, y, VAL, pairs = load_run(run_dir, base_dir); n = len(y)
    print(f"seeds {seeds} | test images {n} | per class", dict(zip(NAMES, np.bincount(y, minlength=4).tolist())))
    names = list(C[seeds[0]].keys())
    rows = []
    for nm in names:
        ss = [s for s in seeds if nm in C[s]]
        acc = [100*(C[s][nm].argmax(1) == y).mean() for s in ss]
        rows.append({"Configuration": nm, "seeds": len(ss), "Accuracy % mean": round(np.mean(acc), 2),
                     "SD": round(np.std(acc, ddof=1), 2) if len(acc) > 1 else np.nan, "min": round(min(acc), 2), "max": round(max(acc), 2),
                     "Macro F1": round(np.mean([mf1(C[s][nm], y) for s in ss]), 4),
                     "ECE": round(np.mean([ece(C[s][nm], y) for s in ss]), 4), "Brier": round(np.mean([brier(C[s][nm], y) for s in ss]), 4)})
    T1 = pd.DataFrame(rows); T1.to_csv(f"{W}/{tag}_table_ablation_baselines.csv", index=False); print(T1.to_string(index=False))
    print("Tuned ResNet weights per seed:", VAL)

    # M4: paired differences, per seed and pooled over seeds (image = unit of analysis)
    comp = [c for c in ["Swin-V2-B, no TTA", "ResNet101, no TTA", "ConvNeXt-T, no TTA", "EfficientNet-B0, no TTA", "ResNet50, no TTA",
                        "Equal fusion, TTA"] if all(c in C[s] for s in seeds)]
    pr = []
    for other in comp:
        for s in seeds:
            a, b = C[s][PRIMARY].argmax(1) == y, C[s][other].argmax(1) == y
            d = a.astype(float) - b; lo, hi = boot_ci(lambda i: 100*d[i].mean(), n)
            n10, n01 = int((a & ~b).sum()), int((~a & b).sum())
            p = binomtest(min(n10, n01), n10 + n01, .5).pvalue if n10 + n01 else 1.0
            pr.append({"Comparison": f"{PRIMARY} minus {other}", "Seed": s, "Difference (points)": round(100*d.mean(), 2),
                       "95% CI": f"{lo:.2f} to {hi:.2f}", "Only primary correct": n10, "Only other correct": n01, "Exact McNemar p": round(p, 4)})
        A = np.mean([C[s][PRIMARY].argmax(1) == y for s in seeds], 0); B = np.mean([C[s][other].argmax(1) == y for s in seeds], 0)
        d = A - B; lo, hi = boot_ci(lambda i: 100*d[i].mean(), n)
        nz = d[d != 0]; flips = RNG.choice([-1, 1], (20000, len(nz))); null = (flips * nz).mean(1) if len(nz) else np.zeros(1)
        p = (np.sum(np.abs(null) >= abs(nz.mean() if len(nz) else 0)) + 1) / (len(null) + 1)
        pr.append({"Comparison": f"{PRIMARY} minus {other}", "Seed": "pooled (images as units)", "Difference (points)": round(100*d.mean(), 2),
                   "95% CI": f"{lo:.2f} to {hi:.2f}", "Only primary correct": "", "Only other correct": "", "Exact McNemar p": f"sign-flip p = {p:.4f}"})
    T2 = pd.DataFrame(pr); T2.to_csv(f"{W}/{tag}_table_paired_differences.csv", index=False); print("\n" + T2.to_string(index=False))

    # M11: calibration differences (primary minus Swin-V2-B), pooled bootstrap over images
    def pooled_metric(fn, nm, i): return np.mean([fn(C[s][nm][i], y[i]) for s in seeds])
    cal = []
    for nm in [PRIMARY, "Swin-V2-B, no TTA", "Equal fusion, TTA", "Swin-V2-B, TTA"]:
        for fn, lab in [(ece, "ECE"), (brier, "Brier")]:
            v = pooled_metric(fn, nm, np.arange(n)); lo, hi = boot_ci(lambda i: pooled_metric(fn, nm, i), n, 1000)
            cal.append({"Configuration": nm, "Metric": lab, "Value (mean over seeds)": round(v, 4), "95% CI": f"{lo:.4f} to {hi:.4f}"})
    for fn, lab in [(ece, "ECE"), (brier, "Brier")]:
        dd = lambda i: pooled_metric(fn, PRIMARY, i) - pooled_metric(fn, "Swin-V2-B, no TTA", i)
        lo, hi = boot_ci(dd, n, 1000)
        cal.append({"Configuration": f"{PRIMARY} minus Swin-V2-B, no TTA", "Metric": lab, "Value (mean over seeds)": round(dd(np.arange(n)), 4), "95% CI": f"{lo:.4f} to {hi:.4f}"})
    T3 = pd.DataFrame(cal); T3.to_csv(f"{W}/{tag}_table_calibration_ci.csv", index=False); print("\n" + T3.to_string(index=False))

    # M7: screening with exact intervals on UNIQUE test images, per seed
    nt_n, tu_n = int((y == NT).sum()), int((y != NT).sum()); sc = []
    for s in seeds:
        pr_ = C[s][PRIMARY].argmax(1); tp = int(((y != NT) & (pr_ != NT)).sum()); tn = int(((y == NT) & (pr_ == NT)).sum())
        a1, a2 = exact_ci(tp, tu_n); b1, b2 = exact_ci(tn, nt_n)
        sc.append({"Seed": s, "Tumor sensitivity": f"{tp}/{tu_n} = {tp/tu_n:.4f} (exact 95% CI {a1:.4f} to {a2:.4f})",
                   "No-tumor specificity": f"{tn}/{nt_n} = {tn/nt_n:.4f} (exact 95% CI {b1:.4f} to {b2:.4f})"})
    T4 = pd.DataFrame(sc); T4.to_csv(f"{W}/{tag}_table_screening_exact.csv", index=False); print("\n" + T4.to_string(index=False))
    print(f"Note: the {tu_n} tumor images are the same in every seed, so intervals use n = {tu_n}, not {tu_n*len(seeds)}.")

    # per-seed confusion matrices of the primary model (M7)
    cms = []
    for s in seeds:
        cm = confusion_matrix(y, C[s][PRIMARY].argmax(1), labels=range(4))
        for i in range(4): cms.append({"Seed": s, "True": NAMES[i], **{f"Pred {NAMES[j]}": int(cm[i, j]) for j in range(4)}})
    pd.DataFrame(cms).to_csv(f"{W}/{tag}_confusion_matrices_per_seed.csv", index=False)

    # seed 42 tables and figures for the primary model
    ref = 42 if 42 in seeds else seeds[0]; pf = C[ref][PRIMARY]; pred = pf.argmax(1); cm = confusion_matrix(y, pred, labels=range(4))
    pc = []
    for i, cname in enumerate(NAMES):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i].sum() - tp; tn = n - tp - fp - fn
        prc = tp/(tp+fp) if tp+fp else 0; rc = tp/(tp+fn); rl, rh = wilson(tp, tp+fn)
        pc.append({"Class": cname, "Precision": round(prc, 4), "Recall": round(rc, 4), "Recall 95% CI": f"{rl:.4f} to {rh:.4f}",
                   "Specificity": round(tn/(tn+fp), 4), "F1": round(2*prc*rc/(prc+rc), 4) if prc + rc else 0})
    yb = label_binarize(y, classes=range(4))
    pc = pd.DataFrame(pc); pc["ROC AUC"] = [round(roc_auc_score(yb[:, i], pf[:, i]), 4) for i in range(4)]
    pc["Average precision"] = [round(average_precision_score(yb[:, i], pf[:, i]), 4) for i in range(4)]
    pc.to_csv(f"{W}/{tag}_table_per_class_primary_seed{ref}.csv", index=False)
    print(f"\nPrimary model ({PRIMARY}), seed {ref}: {np.trace(cm)}/{n} = {100*np.trace(cm)/n:.2f}%\n", pd.DataFrame(cm, index=NAMES, columns=NAMES), "\n", pc.to_string(index=False))
    plt.figure(figsize=(6.8, 5.6)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=NAMES, yticklabels=NAMES, annot_kws={"size": 13})
    plt.xlabel("Predicted label"); plt.ylabel("True label"); plt.title(f"Equal-weight ensemble, no TTA (seed {ref}): {np.trace(cm)}/{n} = {100*np.trace(cm)/n:.2f}%")
    plt.tight_layout(); plt.savefig(f"{W}/{tag}_Fig_CM_primary_seed{ref}.png", dpi=200); plt.close()
    cols = ["black", "blue", "red", "magenta"]
    for kind in ["ROC", "PR"]:
        fig, ax = plt.subplots(figsize=(7.5, 6.2))
        for i in range(4):
            if kind == "ROC":
                a, b, _ = roc_curve(yb[:, i], pf[:, i]); ax.plot(a, b, c=cols[i], lw=2, label=f"{NAMES[i]} (AUC = {pc['ROC AUC'][i]:.4f})")
            else:
                b, a, _ = precision_recall_curve(yb[:, i], pf[:, i]); ax.plot(a, b, c=cols[i], lw=2, label=f"{NAMES[i]} (AP = {pc['Average precision'][i]:.4f})")
        if kind == "ROC": ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        else: ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
        ax.set_title(f"One-versus-rest {kind} curves, equal-weight ensemble, no TTA (seed {ref})"); ax.legend(loc="lower right" if kind == "ROC" else "lower left"); ax.grid(alpha=.35)
        plt.tight_layout(); plt.savefig(f"{W}/{tag}_Fig_{kind}_primary_seed{ref}.png", dpi=200); plt.close()
    return {"seeds": seeds, "C": C, "y": y, "table": T1}

RESULTS = {}
# earlier four-seed near-duplicate run (current paper): this session, an attached saved version, or the seed-42 rerun
cand = [f"{W}/runs_grouped"] + [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/runs_grouped/preds_seed42.npz", recursive=True)] + [f"{W}/runs_grouped_t5"]
GROUPED = next((c for c in cand if glob.glob(f"{c}/preds_seed*.npz")), f"{W}/runs_grouped_t5")
for tag, rd, bd in [("patient", f"{W}/runs_patient", f"{W}/runs_patient_baselines"), ("grouped", GROUPED, None)]:
    if glob.glob(f"{rd}/preds_seed*.npz"): RESULTS[tag] = analyse(rd, bd, tag)
    else: print(f"\n(skipping {tag}: no finished runs in {rd})")

In [ ]:
# ===== M3: does the hash threshold change the result? (seed 42, same training, four grouped splits) =====
import os, numpy as np, pandas as pd
W = "/kaggle/working"
rows = []
for t in [0, 3, 5, 8]:
    f = f"{W}/runs_grouped_t{t}/preds_seed42.npz"
    if not os.path.exists(f): print("missing", f); continue
    d = np.load(f, allow_pickle=True); y = d["test_y"]
    R, S = d["resnet_test_orig"], d["swin_test_orig"]; Rt, St = (R + d["resnet_test_flip"])/2, (S + d["swin_test_flip"])/2
    acc = lambda p: round(100*(p.argmax(1) == y).mean(), 2)
    rows.append({"Hash threshold (bits)": t, "Test images": len(y), "ResNet101": acc(R), "Swin-V2-B": acc(S),
                 "Equal fusion, no TTA": acc((R + S)/2), "Equal fusion, TTA": acc((Rt + St)/2)})
TS = pd.DataFrame(rows)
g = f"{W}/table_threshold_grouping.csv"
if len(TS) and os.path.exists(g):
    G = pd.read_csv(g)
    want = ["Hash threshold (bits)", "Images in those groups", "Groups with mixed labels",
            "Test images with a train/val image within 5 bits", "within 8 bits", "within 12 bits"]
    G = G[[c for c in want if c in G.columns]].rename(columns={
        "Test images with a train/val image within 5 bits": "Test images near train/val (<=5 bits)",
        "within 8 bits": "Test images near train/val (<=8 bits)", "within 12 bits": "Test images near train/val (<=12 bits)"})
    TS = TS.merge(G, on="Hash threshold (bits)", how="left")
TS.to_csv(f"{W}/table_threshold_sensitivity.csv", index=False); print(TS.to_string(index=False))

In [ ]:
# ===== Load the seed-42 models trained on the patient-aware split =====
import torch, importlib, numpy as np, train_seed
importlib.reload(train_seed)
from train_seed import OptimizedResNet101, make_swin, make_convnext, make_resnet50, make_effb0, BrainTumorDataset, test_transform
dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu"); RUN = f"{W}/runs_patient"; BRUN = f"{W}/runs_patient_baselines"
def load(build, f):
    m = build(pretrained=False) if build is not OptimizedResNet101 else OptimizedResNet101(pretrained=False)
    m.load_state_dict(torch.load(f, map_location="cpu")); return m.to(dev).eval()
res = load(OptimizedResNet101, f"{RUN}/resnet_seed42.pt"); swn = load(make_swin, f"{RUN}/swin_seed42.pt")
BASEM = {nm: load(b, f"{BRUN}/{k}_seed42.pt") for k, nm, b in [("convnext", "ConvNeXt-T", make_convnext), ("resnet50", "ResNet50", make_resnet50),
                                                                ("effb0", "EfficientNet-B0", make_effb0)] if os.path.exists(f"{BRUN}/{k}_seed42.pt")}
d42 = np.load(f"{RUN}/preds_seed42.npz", allow_pickle=True)
test_paths, y_test = [str(p) for p in d42["test_paths"]], d42["test_y"]
fused42 = (d42["resnet_test_orig"] + d42["swin_test_orig"]) / 2          # primary model: equal weights, no TTA
pred42 = fused42.argmax(1)
print(f"Loaded seed-42 models on {dev}; primary model accuracy {100*(pred42 == y_test).mean():.2f}% on {len(y_test)} test images; baselines:", list(BASEM))

In [ ]:
# ===== M6: can the no-tumor class be told apart by source or background cues instead of anatomy? =====
import json, numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image, ImageFilter
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score
sp = json.load(open(f"{W}/split_patient.json")); NT = 2
def feats(p):
    im = Image.open(p); w, hgt = im.size; a = np.asarray(im.convert("L"), dtype=np.float32)
    b = max(2, int(0.08 * min(a.shape))); frame = np.concatenate([a[:b].ravel(), a[-b:].ravel(), a[:, :b].ravel(), a[:, -b:].ravel()])
    c = 16; corners = [a[:c, :c].mean(), a[:c, -c:].mean(), a[-c:, :c].mean(), a[-c:, -c:].mean()]
    return [w, hgt, w / hgt, os.path.getsize(p), float(im.mode == "L"), frame.mean(), frame.std(), *corners,
            a.mean(), a.std(), (a < 10).mean(), (a > 245).mean()]
FEAT = ["width", "height", "aspect", "file bytes", "grayscale mode", "frame mean", "frame SD", "corner TL", "corner TR", "corner BL", "corner BR",
        "image mean", "image SD", "fraction near black", "fraction near white"]
tr_p, tr_y = sp["train_paths"] + sp["val_paths"], np.array(sp["train_y"] + sp["val_y"]); te_p, te_y = sp["test_paths"], np.array(sp["test_y"])
Xtr, Xte = np.array([feats(p) for p in tr_p]), np.array([feats(p) for p in te_p])

# (a) what differs between classes, without looking at anatomy
meta = pd.DataFrame(np.vstack([Xtr, Xte]), columns=FEAT); meta["class"] = [NAMES[v] for v in np.concatenate([tr_y, te_y])]
meta["size"] = meta.width.astype(int).astype(str) + "x" + meta.height.astype(int).astype(str)
summ = meta.groupby("class").agg(images=("width", "size"), distinct_sizes=("size", "nunique"),
                                 most_common_size=("size", lambda s: s.value_counts().index[0]),
                                 share_most_common=("size", lambda s: round(s.value_counts(normalize=True).iloc[0], 3)),
                                 median_file_kb=("file bytes", lambda s: round(s.median() / 1024, 1)),
                                 grayscale_share=("grayscale mode", "mean"), median_frame_mean=("frame mean", "median"))
summ.to_csv(f"{W}/table_source_metadata_by_class.csv"); print(summ.to_string())

# (b) an anatomy-free classifier trained on these cues only (train/val -> test, patient-aware split)
out = []
for nm, cols in [("all non-anatomical cues", list(range(len(FEAT)))), ("image size and file size only", [0, 1, 2, 3])]:
    clf = HistGradientBoostingClassifier(random_state=0).fit(Xtr[:, cols], (tr_y == NT).astype(int))
    pr = clf.predict_proba(Xte[:, cols])[:, 1]
    clf4 = HistGradientBoostingClassifier(random_state=0).fit(Xtr[:, cols], tr_y)
    out.append({"Cues used": nm, "No tumor vs tumor ROC AUC": round(roc_auc_score(te_y == NT, pr), 4),
                "No tumor vs tumor balanced accuracy": round(balanced_accuracy_score(te_y == NT, pr > 0.5), 4),
                "4-class accuracy": round(accuracy_score(te_y, clf4.predict(Xte[:, cols])), 4)})
CF = pd.DataFrame(out); CF.to_csv(f"{W}/table_anatomy_free_classifier.csv", index=False); print("\n" + CF.to_string(index=False))

# (c) robustness of the trained models when background cues are removed
def brain_crop(im):
    g = np.asarray(im.convert("L").filter(ImageFilter.GaussianBlur(2)), dtype=np.float32); m = g > 0.1 * g.max()
    ys, xs = np.where(m)
    if len(ys) == 0: return im
    pad = int(0.05 * max(im.size)); box = (max(0, xs.min() - pad), max(0, ys.min() - pad), min(im.width, xs.max() + pad), min(im.height, ys.max() + pad))
    return im.crop(box)
def frame_mask(im, frac=0.12):
    a = np.asarray(im.convert("RGB")).copy(); b = int(frac * min(a.shape[:2])); a[:b] = 0; a[-b:] = 0; a[:, :b] = 0; a[:, -b:] = 0; return Image.fromarray(a)
@torch.no_grad()
def probs(model, paths, fn):
    out = []
    for i in range(0, len(paths), 32):
        x = torch.stack([test_transform(fn(Image.open(p).convert("RGB"))) for p in paths[i:i+32]]).to(dev)
        out.append(F.softmax(model(x), 1).cpu())
    return torch.cat(out).numpy()
rob = []
for vname, fn in [("original", lambda im: im), ("cropped to the brain", brain_crop), ("outer 12% frame blacked out", frame_mask)]:
    pr_, ps_ = probs(res, test_paths, fn), probs(swn, test_paths, fn)
    for mname, p in [("ResNet101", pr_), ("Swin-V2-B", ps_), ("Equal fusion", (pr_ + ps_) / 2)]:
        pd_ = p.argmax(1)
        rob.append({"Input": vname, "Model": mname, "Accuracy %": round(100 * (pd_ == y_test).mean(), 2),
                    "Tumor sensitivity": f"{int(((y_test != NT) & (pd_ != NT)).sum())}/{int((y_test != NT).sum())}",
                    "No-tumor specificity": f"{int(((y_test == NT) & (pd_ == NT)).sum())}/{int((y_test == NT).sum())}"})
RB = pd.DataFrame(rob); RB.to_csv(f"{W}/table_background_robustness.csv", index=False); print("\n" + RB.to_string(index=False))

In [ ]:
# ===== M1: Figure 11 (t-SNE) and Figure 12 (error grid) for the primary model (equal fusion, no TTA, seed 42) =====
import numpy as np, torch, matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from PIL import Image
feats_ = {}
h1 = res.out.register_forward_hook(lambda m, i, o: feats_.__setitem__("r", i[0].detach().cpu()))
h2 = swn.head.register_forward_hook(lambda m, i, o: feats_.__setitem__("s", i[0].detach().cpu()))
dl = torch.utils.data.DataLoader(BrainTumorDataset(test_paths, [int(v) for v in y_test], test_transform), batch_size=32, num_workers=2)
FR, FS = [], []
with torch.no_grad():
    for x, _ in dl:
        x = x.to(dev); res(x); swn(x); FR.append(feats_["r"]); FS.append(feats_["s"])
h1.remove(); h2.remove()
z = lambda a: (a - a.mean(0)) / (a.std(0) + 1e-6)
X = np.concatenate([z(torch.cat(FR).numpy()), z(torch.cat(FS).numpy())], 1)
emb = TSNE(n_components=2, perplexity=30, init="pca", random_state=42).fit_transform(X)
wrong = pred42 != y_test; cols = ["black", "blue", "red", "magenta"]
plt.figure(figsize=(8, 6.5))
for c in range(4):
    m = (y_test == c) & ~wrong; plt.scatter(emb[m, 0], emb[m, 1], s=12, c=cols[c], alpha=.6, label=NAMES[c])
for c in range(4):
    m = (y_test == c) & wrong; plt.scatter(emb[m, 0], emb[m, 1], s=70, c=cols[c], marker="x", linewidths=2)
plt.scatter([], [], marker="x", c="gray", s=70, label="Misclassified (colour = true class)")
plt.legend(); plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2"); plt.title("t-SNE of standardized, concatenated pre-classifier features (seed 42)")
plt.tight_layout(); plt.savefig(f"{W}/Fig11_tSNE_primary_seed42.png", dpi=300, bbox_inches="tight"); plt.show()
types = {}
for i in np.where(wrong)[0]: types.setdefault((int(y_test[i]), int(pred42[i])), []).append(i)
print("Errors of the primary model:", int(wrong.sum()), {f"{NAMES[a]} -> {NAMES[b]}": len(v) for (a, b), v in sorted(types.items(), key=lambda t: -len(t[1]))})
show, lists = [], [list(v) for _, v in sorted(types.items(), key=lambda t: -len(t[1]))]
while len(show) < 12 and any(lists):
    for L in lists:
        if L and len(show) < 12: show.append(L.pop(0))
rows_ = int(np.ceil(len(show) / 4)) or 1
fig, ax = plt.subplots(rows_, 4, figsize=(12.8, 3.5 * rows_), squeeze=False)
for k, a in enumerate(ax.flat):
    a.axis("off")
    if k < len(show):
        i = show[k]; a.imshow(Image.open(test_paths[i]).convert("L"), cmap="gray")
        a.set_title(f"True: {NAMES[y_test[i]]}\nPredicted: {NAMES[pred42[i]]} (p = {fused42[i].max():.2f})", fontsize=9)
plt.tight_layout(); plt.savefig(f"{W}/Fig12_errors_primary_seed42.png", dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# ===== M12: Grad-CAM on random and failed cases, lesion localization against Figshare masks, sanity check =====
!pip install -q grad-cam
import os, copy, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import spearmanr
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
W = "/kaggle/working"; NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
assert "res" in globals() and "swn" in globals(), "Run the 'Load the seed-42 models' cell first, then this cell."

# ResNet101: last convolutional stage (layer4, 7 x 7). Swin-V2-B: output of the last stage, whose tokens come as
# (batch, 7, 7, 1024); they are permuted to (batch, 1024, 7, 7) so Grad-CAM can weight channels as for a CNN.
swin_reshape = lambda t: t.permute(0, 3, 1, 2)
def cams_for(model, layer, reshape=None):
    return GradCAM(model=model, target_layers=[layer], reshape_transform=reshape)
cam_r, cam_s = cams_for(res, res.features[-1]), cams_for(swn, swn.features[-1], swin_reshape)
MEAN_, STD_ = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
def tensor(i): return test_transform(Image.open(test_paths[i]).convert("RGB")).unsqueeze(0).to(dev)
def to_img(x): return (x[0].cpu().numpy().transpose(1, 2, 0) * STD_ + MEAN_).clip(0, 1).astype(np.float32)
def cam(c, x, cls): return c(input_tensor=x, targets=[ClassifierOutputTarget(int(cls))])[0]
pr_r, pr_s = d42["resnet_test_orig"].argmax(1), d42["swin_test_orig"].argmax(1)

def panel(idx, fname, title):
    fig, ax = plt.subplots(len(idx), 3, figsize=(9.6, 3.2 * len(idx)), squeeze=False)
    for r, i in enumerate(idx):
        x = tensor(i); im = to_img(x); cls = pred42[i]
        ax[r, 0].imshow(im); ax[r, 0].set_title(f"True: {NAMES[y_test[i]]} | fusion: {NAMES[pred42[i]]}", fontsize=8)
        ax[r, 1].imshow(show_cam_on_image(im, cam(cam_r, x, cls), use_rgb=True)); ax[r, 1].set_title(f"ResNet101 layer4 (pred {NAMES[pr_r[i]]})", fontsize=8)
        ax[r, 2].imshow(show_cam_on_image(im, cam(cam_s, x, cls), use_rgb=True)); ax[r, 2].set_title(f"Swin-V2-B last stage (pred {NAMES[pr_s[i]]})", fontsize=8)
        for a in ax[r]: a.axis("off")
    fig.suptitle(title + "  [maps for the class predicted by the fusion; colour scale 0 to 1]", fontsize=9)
    plt.tight_layout(); plt.savefig(f"{W}/{fname}", dpi=250, bbox_inches="tight"); plt.show()
rng = np.random.default_rng(0)
rand_idx = rng.choice(len(y_test), 8, replace=False)
fail_idx = np.where(pred42 != y_test)[0][:4]
panel(rand_idx, "Fig_GradCAM_random_seed42.png", "Eight randomly chosen test images (seed 0)")
if len(fail_idx): panel(fail_idx, "Fig_GradCAM_failures_seed42.png", "Test images the fusion got wrong")

# lesion localization: test tumor images that match a Figshare image within 2 bits (same framing), true-class maps
mp = pd.read_csv(f"{W}/sartaj_all_to_figshare_pid.csv", dtype={"pid": str}); tail = lambda p: "/".join(str(p).split("/")[-3:])
mk = ({tail(r.path): r.fig_mask for r in mp.itertuples() if isinstance(r.fig_mask, str) and r.fig_dist <= 2}
      if "fig_mask" in mp.columns else {})
if not mk: print("No mask paths in the mapping file (it predates this notebook): delete sartaj_all_to_figshare_pid.csv and rerun the mapping cell for localization.")
loc_idx = [i for i, p in enumerate(test_paths) if y_test[i] != 2 and tail(p) in mk]
def mask224(f):
    return test_transform.transforms[1](Image.open(f).convert("L").resize((256, 256), Image.NEAREST))   # resize 256 + centre crop 224, as the image
rec = []
for i in loc_idx:
    m = np.asarray(mask224(mk[tail(test_paths[i])])) > 127
    if m.sum() == 0: continue
    x = tensor(i)
    for nm, c in [("ResNet101", cam_r), ("Swin-V2-B", cam_s)]:
        g = cam(c, x, y_test[i]); hit = bool(m.flat[np.argmax(g)])
        rec.append({"image": i, "branch": nm, "pointing hit": hit, "energy in mask": float((g * m).sum() / max(g.sum(), 1e-8)), "mask area": float(m.mean())})
LOC = pd.DataFrame(rec)
if len(LOC):
    summ = LOC.groupby("branch").agg(images=("image", "nunique"), pointing_game=("pointing hit", "mean"), energy_in_mask=("energy in mask", "mean"),
                                     chance_level_mask_area=("mask area", "mean")).round(4)
    summ.to_csv(f"{W}/table_gradcam_localization.csv"); print(f"Lesion localization on {LOC.image.nunique()} masked tumor test images:\n", summ.to_string())
    ex = loc_idx[:4]                                         # alignment check: mask outline on the image
    fig, ax = plt.subplots(1, len(ex), figsize=(3.2 * len(ex), 3.2), squeeze=False)
    for a, i in zip(ax[0], ex):
        a.imshow(to_img(tensor(i))); a.contour(np.asarray(mask224(mk[tail(test_paths[i])])) > 127, colors="lime", linewidths=1); a.axis("off")
        a.set_title(NAMES[y_test[i]], fontsize=8)
    plt.tight_layout(); plt.savefig(f"{W}/Fig_mask_alignment_check.png", dpi=200); plt.show()
else: print("No test images matched to Figshare masks.")

# sanity check (model-parameter randomization): maps should change when the explained stage is re-initialised
def randomized(model, stage):
    m2 = copy.deepcopy(model); st = stage(m2)
    for mod in st.modules():
        if hasattr(mod, "reset_parameters"): mod.reset_parameters()
    return m2.eval()
r_rand, s_rand = randomized(res, lambda m: m.features[-1]), randomized(swn, lambda m: m.features[-1])
cr2, cs2 = cams_for(r_rand, r_rand.features[-1]), cams_for(s_rand, s_rand.features[-1], swin_reshape)
sc = []
for i in rng.choice(len(y_test), 24, replace=False):
    x = tensor(i); a, b, a2, b2 = cam(cam_r, x, pred42[i]), cam(cam_s, x, pred42[i]), cam(cr2, x, pred42[i]), cam(cs2, x, pred42[i])
    ds = lambda g: g[::4, ::4].ravel()
    sc.append({"ResNet101 vs randomized": spearmanr(ds(a), ds(a2))[0], "Swin-V2-B vs randomized": spearmanr(ds(b), ds(b2))[0],
               "ResNet101 vs Swin-V2-B": spearmanr(ds(a), ds(b))[0]})
SC = pd.DataFrame(sc).agg(["mean", "std"]).round(3); SC.to_csv(f"{W}/table_gradcam_sanity.csv")
print("\nSpearman correlation of maps (24 random test images). Low values against randomized weights = the maps depend on learned weights:\n", SC.to_string())

In [ ]:
# ===== M17: inference time and model size, with the full protocol stated =====
import time, json, platform, numpy as np, pandas as pd, torch
env = {"GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "PyTorch": torch.__version__,
       "CUDA (PyTorch build)": torch.version.cuda, "cuDNN": torch.backends.cudnn.version(), "Python": platform.python_version(),
       "precision": "fp32", "cudnn.benchmark": torch.backends.cudnn.benchmark}
print(env)
WARM, REPS, REPEATS = 10, 50, 3
def time_it(fn, bs):
    x = torch.randn(bs, 3, 224, 224, device=dev)      # input already on the GPU: preprocessing and data loading are excluded
    out = []
    with torch.no_grad():
        for _ in range(WARM): fn(x)
        for _ in range(REPEATS):
            torch.cuda.synchronize(); t = time.perf_counter()
            for _ in range(REPS): fn(x)
            torch.cuda.synchronize(); out.append((time.perf_counter() - t) / REPS * 1000 / bs)
    return np.mean(out), np.std(out)
flip = lambda m: (lambda x: (m(x).softmax(1) + m(torch.flip(x, [3])).softmax(1)) / 2)
MODELS = {"ResNet101": res, "Swin-V2-B": swn,
          "Equal fusion, no TTA": lambda x: (res(x).softmax(1) + swn(x).softmax(1)) / 2,
          "Equal fusion, TTA": lambda x: (flip(res)(x) + flip(swn)(x)) / 2, **BASEM}
NPAR = {"ResNet101": res, "Swin-V2-B": swn, **BASEM}
rows = []
for nm, f in MODELS.items():
    p = sum(q.numel() for q in NPAR[nm].parameters()) / 1e6 if nm in NPAR else (sum(q.numel() for q in res.parameters()) + sum(q.numel() for q in swn.parameters())) / 1e6
    r = {"Model": nm, "Parameters (M)": round(p, 2)}
    for bs in (1, 16):
        m_, s_ = time_it(f, bs); r[f"ms per image, batch {bs}"] = f"{m_:.2f} ± {s_:.2f}"
    rows.append(r)
TM = pd.DataFrame(rows); TM.to_csv(f"{W}/table_timing_review.csv", index=False); print(TM.to_string(index=False))
json.dump({**env, "warm-up passes": WARM, "timed passes": REPS, "repeats": REPEATS, "synchronised": True, "preprocessing included": False},
          open(f"{W}/timing_protocol.json", "w"), indent=1)
print(f"Protocol: {WARM} warm-up passes, then {REPEATS} repeats of {REPS} timed passes with torch.cuda.synchronize() before and after; "
      "values are mean ± SD over repeats; random input already on the GPU (preprocessing excluded); one GPU; eval mode, no_grad.")

In [ ]:
# ===== M10: reproducibility package + one results zip (no model weights, no Figshare images) =====
import os, glob, json, zipfile, subprocess, shutil, numpy as np, pandas as pd, torch, torchvision
PK = f"{W}/repro_package"; os.makedirs(PK, exist_ok=True)
for f in ["train_seed.py", "split_patient.json", "split_grouped_t0.json", "split_grouped_t3.json", "split_grouped_t5.json", "split_grouped_t8.json",
          "image_table_hash_groups_splits.csv", "sartaj_all_to_figshare_pid.csv", "timing_protocol.json"]:
    if os.path.exists(f"{W}/{f}"): shutil.copy(f"{W}/{f}", PK)
# per-image predictions for every seed, model and view
rec = []
for d_, models_ in [(f"{W}/runs_patient", ["resnet", "swin"]), (f"{W}/runs_patient_baselines", ["convnext", "resnet50", "effb0"])]:
    for f in sorted(glob.glob(f"{d_}/probs_*_seed*.npz")):
        m = os.path.basename(f).split("_")[1]; s = int(f.split("seed")[-1].split(".")[0]); z = np.load(f, allow_pickle=True)
        for view in ["test_orig", "test_flip"]:
            for i, (p, yy) in enumerate(zip(z["test_paths"], z["test_y"])):
                rec.append({"file": "/".join(str(p).split("/")[-3:]), "label": int(yy), "model": m, "seed": s, "view": view.split("_")[1],
                            **{f"p_{NAMES[c]}": round(float(z[view][i, c]), 6) for c in range(4)}})
pd.DataFrame(rec).to_csv(f"{PK}/test_predictions_all_seeds.csv.gz", index=False, compression="gzip")
cfg = {"split": "stratified group split 70/10/20, SPLIT_SEED 42; groups = perceptual hash within 5 bits OR same Figshare patient",
       "seeds": [42, 43, 44, 45], "epochs": 12, "batch size": 16, "optimizer": "AdamW, weight decay 1e-4",
       "learning rate": {"ResNet101": 1e-4, "Swin-V2-B": 5e-5, "ConvNeXt-T": 1e-4, "ResNet50": 1e-4, "EfficientNet-B0": 1e-4},
       "scheduler": "CosineAnnealingLR, T_max = 12 epochs, eta_min = 0, stepped once per epoch",
       "loss": "cross-entropy with label smoothing 0.1, no class weighting",
       "input": "image opened with PIL and converted to RGB (grayscale copied to three channels); resize 256x256, centre crop 224x224, ImageNet mean/std",
       "training augmentation (torchvision, in this order)": "Resize(256,256), CenterCrop(224), RandomHorizontalFlip(0.5), RandomRotation(10), ToTensor, Normalize",
       "batch normalisation": "standard train/eval modes; statistics updated during training only",
       "checkpoint selection": "highest validation accuracy; ties keep the earlier epoch (strict >)",
       "determinism": "random, numpy and torch seeded; cudnn.deterministic = True, cudnn.benchmark = False; seeded DataLoader generator and workers",
       "weights": {"ResNet101": "torchvision IMAGENET1K_V2", "Swin-V2-B": "torchvision swin_v2_b IMAGENET1K_V1", "ConvNeXt-T": "torchvision IMAGENET1K_V1",
                   "ResNet50": "torchvision IMAGENET1K_V2", "EfficientNet-B0": "torchvision IMAGENET1K_V1"},
       "versions": {"torch": torch.__version__, "torchvision": torchvision.__version__, "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version()}}
json.dump(cfg, open(f"{PK}/training_config.json", "w"), indent=1)
open(f"{PK}/environment.txt", "w").write(subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout)
open(f"{PK}/README.md", "w").write("""# Leakage-controlled ResNet101 + Swin-V2-B brain tumor MRI classification

Data: Kaggle "Brain Tumor Classification (MRI)" (Bhuvaji et al., 2020) and Figshare brain tumor dataset (Cheng et al.) for patient IDs and masks.
Files: train_seed.py (training), split_*.json (exact train/val/test file lists), image_table_hash_groups_splits.csv (hash, group IDs,
patient IDs, split membership for every image), test_predictions_all_seeds.csv.gz (per-image probabilities for every model, seed and view),
training_config.json, environment.txt. Model checkpoints are archived separately (see the paper's data availability statement).
""")
summary = []
for f in sorted(glob.glob(f"{W}/*table*.csv")) + sorted(glob.glob(f"{W}/*.json")):
    if os.path.basename(f).startswith("split_") or os.path.basename(f).startswith("jobs_"): continue
    try: summary.append(f"\n===== {os.path.basename(f)} =====\n" + (pd.read_csv(f).to_string(index=False) if f.endswith(".csv") else open(f).read()))
    except Exception: pass
fails = [l for f in glob.glob(f"{W}/train_gpu*.out") for l in open(f) if "FAILED" in l]
summary.append("\n===== Failures =====\n" + ("\n".join(fails) if fails else "none"))
open(f"{W}/results_summary_review2.txt", "w").write("\n".join(summary))
zp = f"{W}/results_review2.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{W}/*.csv") + glob.glob(f"{W}/*.png") + glob.glob(f"{W}/*.json") + glob.glob(f"{W}/*.txt") + glob.glob(f"{W}/*.out"):
        z.write(f, os.path.basename(f))
    for f in glob.glob(f"{PK}/*"): z.write(f, "repro_package/" + os.path.basename(f))
    for d_ in ["runs_patient", "runs_patient_baselines", "runs_grouped_t0", "runs_grouped_t3", "runs_grouped_t5", "runs_grouped_t8"]:
        for f in glob.glob(f"{W}/{d_}/*.npz") + glob.glob(f"{W}/{d_}/*.json") + glob.glob(f"{W}/{d_}/*.txt"): z.write(f, f"{d_}/{os.path.basename(f)}")
print(f"Saved {zp} ({os.path.getsize(zp)/2**20:.1f} MB) and results_summary_review2.txt")
try:
    from IPython.display import FileLink, display
    os.chdir(W); display(FileLink("results_review2.zip")); display(FileLink("results_summary_review2.txt"))
except Exception: print("Download both files from the Output panel (/kaggle/working).")

In [ ]:
# ===== M12: does Grad-CAM land on the tumor? Lesion localization against Figshare tumor masks (seed-42 models) =====
!pip install -q grad-cam imagehash h5py
import os, glob, json, importlib, numpy as np, pandas as pd, torch, imagehash, matplotlib.pyplot as plt, train_seed
from PIL import Image
from scipy.stats import binomtest
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
W = "/kaggle/working"; NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
importlib.reload(train_seed); from train_seed import OptimizedResNet101, make_swin, test_transform
dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 1. tumor masks and image hashes from the original Figshare .mat files (cached after the first run)
cache = f"{W}/figshare/mask_index.csv"; MSK = f"{W}/figshare/mask"; os.makedirs(MSK, exist_ok=True)
if os.path.exists(cache):
    fx = pd.read_csv(cache)
else:
    mats = sorted({m for m in glob.glob(f"{W}/figshare/**/*.mat", recursive=True) + glob.glob("/kaggle/input/**/*.mat", recursive=True)
                   if "cvind" not in os.path.basename(m).lower()})
    assert len(mats) >= 3000, "Figshare .mat files not found: run the Figshare download cell once, then run this cell again."
    def read_mat(f):
        try:
            import h5py
            with h5py.File(f, "r") as h:
                cj = h["cjdata"]; img = np.array(cj["image"]).T; msk = np.array(cj["tumorMask"]).T
        except OSError:
            from scipy.io import loadmat
            cj = loadmat(f)["cjdata"][0, 0]; img = cj["image"]; msk = cj["tumorMask"]
        img = img.astype(np.float32); img = ((img - img.min()) / max(img.max() - img.min(), 1e-6) * 255).astype(np.uint8)
        return img, (np.asarray(msk) > 0).astype(np.uint8) * 255
    rec = []
    for k, f in enumerate(mats):
        img, msk = read_mat(f); mp = f"{MSK}/{os.path.splitext(os.path.basename(f))[0]}.png"; Image.fromarray(msk).save(mp)
        rec.append({"mask": mp, "hash": str(imagehash.phash(Image.fromarray(img).convert("L")))})
        if k % 500 == 0: print(f"  read {k}/{len(mats)} Figshare files")
    fx = pd.DataFrame(rec); fx.to_csv(cache, index=False)
hf = np.array([int(h, 16) for h in fx["hash"]], dtype=np.uint64)
print(f"Figshare masks available: {len(fx)}")

# 2. seed-42 models from the patient-aware run
RUN = f"{W}/runs_patient"
res = OptimizedResNet101(pretrained=False); res.load_state_dict(torch.load(f"{RUN}/resnet_seed42.pt", map_location="cpu")); res.to(dev).eval()
swn = make_swin(pretrained=False); swn.load_state_dict(torch.load(f"{RUN}/swin_seed42.pt", map_location="cpu")); swn.to(dev).eval()
cam_r = GradCAM(model=res, target_layers=[res.features[-1]])
cam_s = GradCAM(model=swn, target_layers=[swn.features[-1]], reshape_transform=lambda t: t.permute(0, 3, 1, 2))

# 3. test tumor images that are copies of a Figshare image (within 2 bits, so the framing is the same)
sp = json.load(open(f"{W}/split_patient.json")); test_paths, y = sp["test_paths"], np.array(sp["test_y"])
ht = np.array([int(str(imagehash.phash(Image.open(p).convert("L"))), 16) for p in test_paths], dtype=np.uint64)
dist = np.unpackbits((ht[:, None] ^ hf[None, :]).view(np.uint8).reshape(len(ht), len(hf), 8), axis=2).sum(2)
best, near = dist.argmin(1), dist.min(1)
sel = [i for i in range(len(y)) if y[i] != 2 and near[i] <= 2]
print(f"Tumor test images matched to a Figshare mask: {len(sel)} of {int((y != 2).sum())}")
def mask224(f):     # same geometry as the image: resize to 256 x 256, then centre crop 224
    return np.asarray(test_transform.transforms[1](Image.open(f).convert("L").resize((256, 256), Image.NEAREST))) > 127
def tensor(p): return test_transform(Image.open(p).convert("RGB")).unsqueeze(0).to(dev)

rec = []
for i in sel:
    m = mask224(fx["mask"][best[i]])
    if m.sum() == 0: continue
    x = tensor(test_paths[i])
    for nm, c in [("ResNet101", cam_r), ("Swin-V2-B", cam_s)]:
        g = c(input_tensor=x, targets=[ClassifierOutputTarget(int(y[i]))])[0]          # map for the TRUE class
        rec.append({"image": i, "class": NAMES[y[i]], "branch": nm, "pointing hit": bool(m.flat[np.argmax(g)]),
                    "energy in mask": float((g * m).sum() / max(g.sum(), 1e-8)), "mask area": float(m.mean())})
LOC = pd.DataFrame(rec)
rows = []
for nm, g_ in LOC.groupby("branch"):
    k, n = int(g_["pointing hit"].sum()), len(g_); ci = binomtest(k, n).proportion_ci(method="exact")
    rows.append({"Branch": nm, "Images": n, "Pointing game": f"{k}/{n} = {k/n:.3f} (95% CI {ci.low:.3f} to {ci.high:.3f})",
                 "Energy inside mask (mean)": round(g_["energy in mask"].mean(), 3), "Chance level (mean mask area)": round(g_["mask area"].mean(), 3)})
T = pd.DataFrame(rows); T.to_csv(f"{W}/table_gradcam_localization.csv", index=False); print("\n" + T.to_string(index=False))
LOC.groupby(["branch", "class"])[["pointing hit", "energy in mask", "mask area"]].mean().round(3).to_csv(f"{W}/table_gradcam_localization_by_class.csv")
print("\nBy class:\n", pd.read_csv(f"{W}/table_gradcam_localization_by_class.csv").to_string(index=False))

# 4. alignment check: the mask outline must sit on the tumor; if not, the numbers above are not valid
MEAN_, STD_ = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
ex = sel[:6]
fig, ax = plt.subplots(1, len(ex), figsize=(3 * len(ex), 3.2), squeeze=False)
for a, i in zip(ax[0], ex):
    im = (tensor(test_paths[i])[0].cpu().numpy().transpose(1, 2, 0) * STD_ + MEAN_).clip(0, 1)
    a.imshow(im); a.contour(mask224(fx["mask"][best[i]]), colors="lime", linewidths=1); a.set_title(NAMES[y[i]], fontsize=8); a.axis("off")
plt.tight_layout(); plt.savefig(f"{W}/Fig_mask_alignment_check.png", dpi=200); plt.show()

In [ ]:
# ===== Source confounding on the patient-aware split: do the trained models follow source cues? =====
import os, json, numpy as np, pandas as pd
from PIL import Image
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier
W = "/kaggle/working"; NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]; NT = 2
sp = json.load(open(f"{W}/split_patient.json"))
def feats(p):
    im = Image.open(p); w, hgt = im.size; a = np.asarray(im.convert("L"), dtype=np.float32)
    b = max(2, int(0.08 * min(a.shape))); frame = np.concatenate([a[:b].ravel(), a[-b:].ravel(), a[:, :b].ravel(), a[:, -b:].ravel()])
    c = 16; corners = [a[:c, :c].mean(), a[:c, -c:].mean(), a[-c:, :c].mean(), a[-c:, -c:].mean()]
    return [w, hgt, w / hgt, os.path.getsize(p), float(im.mode == "L"), frame.mean(), frame.std(), *corners, a.mean(), a.std(), (a < 10).mean(), (a > 245).mean()]
tr_p, tr_y = sp["train_paths"] + sp["val_paths"], np.array(sp["train_y"] + sp["val_y"]); te_p, y = sp["test_paths"], np.array(sp["test_y"])
Xtr, Xte = np.array([feats(p) for p in tr_p]), np.array([feats(p) for p in te_p])
shortcut = HistGradientBoostingClassifier(random_state=0).fit(Xtr, (tr_y == NT).astype(int)).predict_proba(Xte)[:, 1]   # P(no tumor) from source cues only
size512 = (Xte[:, 0] == 512) & (Xte[:, 1] == 512)

seeds = [42, 43, 44, 45]; fused = {}
for s in seeds:
    d = np.load(f"{W}/runs_patient/preds_seed{s}.npz", allow_pickle=True); assert list(map(str, d["test_paths"])) == te_p
    fused[s] = (d["resnet_test_orig"] + d["swin_test_orig"]) / 2
pnt = np.mean([fused[s][:, NT] for s in seeds], 0)                        # model's P(no tumor), mean over seeds
says_nt = {s: fused[s].argmax(1) == NT for s in seeds}

# 1. results split by image size (512 x 512 is typical of the tumor sources)
rows = []
for cls_name, mcls in [("No tumor", y == NT), ("Tumor (any)", y != NT)]:
    for grp, mg in [("512 x 512", size512), ("other size", ~size512)]:
        m = mcls & mg
        if m.sum() == 0: continue
        if cls_name == "No tumor": vals = [100 * says_nt[s][m].mean() for s in seeds]; lab = "called no tumor (specificity) %"
        else: vals = [100 * (~says_nt[s][m]).mean() for s in seeds]; lab = "called tumor (sensitivity) %"
        rows.append({"True class": cls_name, "Image size": grp, "Images": int(m.sum()), "Measure": lab,
                     "Mean over seeds": round(np.mean(vals), 2), "Per seed": [round(v, 1) for v in vals]})
T1 = pd.DataFrame(rows); T1.to_csv(f"{W}/table_confound_by_size.csv", index=False); print(T1.to_string(index=False))

# 2. images where the source cues point the WRONG way: does the model follow them?
rows = []
for cls_name, mcls, wrong_cue in [("No tumor", y == NT, shortcut < 0.5), ("Tumor (any)", y != NT, shortcut >= 0.5)]:
    for cue, mc in [("source cues misleading", mcls & wrong_cue), ("source cues consistent", mcls & ~wrong_cue)]:
        if mc.sum() == 0: continue
        ok = [((says_nt[s] if cls_name == "No tumor" else ~says_nt[s])[mc]).mean() * 100 for s in seeds]
        rows.append({"True class": cls_name, "Source-only classifier": cue, "Images": int(mc.sum()), "Model correct % (mean over seeds)": round(np.mean(ok), 2)})
T2 = pd.DataFrame(rows); T2.to_csv(f"{W}/table_confound_misleading_cues.csv", index=False); print("\n" + T2.to_string(index=False))

# 3. within each true class, does the model's no-tumor probability track the source-only probability?
rho_nt = spearmanr(shortcut[y == NT], pnt[y == NT]); rho_t = spearmanr(shortcut[y != NT], pnt[y != NT])
print(f"\nSpearman correlation between source-only P(no tumor) and model P(no tumor):"
      f"\n  within no-tumor images: rho = {rho_nt[0]:.3f} (p = {rho_nt[1]:.3g}, n = {int((y == NT).sum())})"
      f"\n  within tumor images:    rho = {rho_t[0]:.3f} (p = {rho_t[1]:.3g}, n = {int((y != NT).sum())})")
json.dump({"rho_within_no_tumor": rho_nt[0], "p_within_no_tumor": rho_nt[1], "rho_within_tumor": rho_t[0], "p_within_tumor": rho_t[1]},
          open(f"{W}/confound_correlation.json", "w"), indent=1)

In [ ]:
# ===== External dataset (BDNeuro-MRI): overlap check, source check, external test (no retraining) =====
!pip install -q imagehash
import os, glob, json, zipfile, importlib, numpy as np, pandas as pd, matplotlib.pyplot as plt, imagehash, torch, torch.nn.functional as F
from PIL import Image
from math import sqrt
from scipy.stats import binomtest
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix
W = "/kaggle/working"; OUT = f"{W}/external_bdneuro"; os.makedirs(OUT, exist_ok=True)
NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]; NT = 2
CLASS_OF = {"glioma": 0, "meningioma": 1, "no_tumor": 2, "notumor": 2, "no tumor": 2, "normal": 2, "pituitary": 3}
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

# ---------- 0. find the attached BDNeuro-MRI files (extract a zip if Kaggle did not) ----------
def find_bdneuro():
    for root in glob.glob("/kaggle/input/**/", recursive=True):
        low = root.lower()
        if "sartaj" in low or "brain-tumor-classification-mri" in low: continue
        if os.path.isdir(os.path.join(root, "test", "glioma")) and os.path.isdir(os.path.join(root, "train", "glioma")): return root
    return None
EXT = find_bdneuro()
if EXT is None:
    zips = [z for z in glob.glob("/kaggle/input/**/*.zip", recursive=True)]
    for z in zips:
        with zipfile.ZipFile(z) as zf: zf.extractall(f"{W}/bdneuro_extracted")
    for root in glob.glob(f"{W}/bdneuro_extracted/**/", recursive=True):
        if os.path.isdir(os.path.join(root, "test", "glioma")): EXT = root; break
assert EXT, "BDNeuro-MRI not found: attach your Kaggle copy with Add Input, then run this cell again."
ext_paths, ext_y, ext_part = [], [], []
for part in ["train", "val", "test"]:
    for d in sorted(glob.glob(os.path.join(EXT, part, "*"))):
        c = CLASS_OF.get(os.path.basename(d).lower().replace("-", "_"))
        if c is None: print("skipping unknown folder", d); continue
        for f in sorted(glob.glob(os.path.join(d, "*"))):
            if f.lower().endswith(IMG_EXT): ext_paths.append(f); ext_y.append(c); ext_part.append(part)
ext_y, ext_part = np.array(ext_y), np.array(ext_part)
print(f"BDNeuro-MRI found at {EXT}: {len(ext_paths)} images |", dict(zip(NAMES, np.bincount(ext_y, minlength=4).tolist())))

# ---------- 1. overlap with Sartaj and Figshare ----------
assert os.path.exists(f"{W}/train_seed.py"), "train_seed.py missing: run the %%writefile train_seed.py cell first."
import train_seed; importlib.reload(train_seed)
from train_seed import all_images, OptimizedResNet101, make_swin, make_convnext, make_resnet50, make_effb0, test_transform
ph = lambda p: int(str(imagehash.phash(Image.open(p).convert("L"))), 16)
s_paths, s_lab = all_images()
h_s = np.load(f"{W}/phash_all.npy") if os.path.exists(f"{W}/phash_all.npy") else np.array([ph(p) for p in s_paths], dtype=np.uint64)
if os.path.exists(f"{W}/figshare/mask_index.csv"):
    h_f = np.array([int(x, 16) for x in pd.read_csv(f"{W}/figshare/mask_index.csv")["hash"]], dtype=np.uint64); f_names = ["figshare"] * len(h_f)
else:
    fp = sorted(glob.glob(f"{W}/figshare/jpg/*.png")); h_f = np.array([ph(p) for p in fp], dtype=np.uint64); f_names = fp
    if not len(fp): print("WARNING: no Figshare hashes found; run the Figshare cell for the full overlap check.")
h_e = np.array([ph(p) for p in ext_paths], dtype=np.uint64)
def nearest(a, b):
    if len(b) == 0: return np.full(len(a), 64), np.zeros(len(a), int)
    dm, im = np.empty(len(a), int), np.empty(len(a), int)
    for i in range(0, len(a), 256):
        d = np.unpackbits((a[i:i+256, None] ^ b[None, :]).view(np.uint8).reshape(len(a[i:i+256]), len(b), 8), axis=2).sum(2)
        dm[i:i+256], im[i:i+256] = d.min(1), d.argmin(1)
    return dm, im
d_s, i_s = nearest(h_e, h_s); d_f, i_f = nearest(h_e, h_f)
rows = []
for c in range(4):
    m = ext_y == c
    rows.append({"Class": NAMES[c], "Images": int(m.sum()),
                 "Exact match to Sartaj (0 bits)": int((m & (d_s == 0)).sum()), "Sartaj within 5 bits": int((m & (d_s <= 5)).sum()),
                 "Sartaj within 8 bits": int((m & (d_s <= 8)).sum()), "Exact match to Figshare": int((m & (d_f == 0)).sum()),
                 "Figshare within 5 bits": int((m & (d_f <= 5)).sum()), "Figshare within 8 bits": int((m & (d_f <= 8)).sum())})
OV = pd.DataFrame(rows); OV.loc[len(OV)] = ["Total"] + OV.iloc[:, 1:].sum().tolist()
OV.to_csv(f"{OUT}/table_overlap.csv", index=False); print("\n" + OV.to_string(index=False))
overlap = (d_s <= 5) | (d_f <= 5)
pd.DataFrame({"bdneuro_image": ext_paths, "label": [NAMES[v] for v in ext_y], "split": ext_part,
              "nearest_sartaj": [s_paths[j] for j in i_s], "dist_sartaj": d_s, "dist_figshare": d_f}).to_csv(f"{OUT}/overlap_per_image.csv", index=False)
show = np.argsort(d_s)[:6]
fig, ax = plt.subplots(len(show), 2, figsize=(6, 3 * len(show)), squeeze=False)
for r, i in enumerate(show):
    for c_, (p, t) in enumerate([(ext_paths[i], f"BDNeuro: {NAMES[ext_y[i]]}"), (s_paths[i_s[i]], f"Sartaj: {NAMES[s_lab[i_s[i]]]}")]):
        ax[r, c_].imshow(Image.open(p).convert("L"), cmap="gray"); ax[r, c_].axis("off"); ax[r, c_].set_title(f"{t} | {d_s[i]} bits", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/Fig_closest_sartaj_matches.png", dpi=150); plt.show()
print(f"\nImages kept for the external test (no Sartaj or Figshare image within 5 bits): {int((~overlap).sum())} of {len(ext_paths)}")

# ---------- 2. source check ----------
def feats(p):
    im = Image.open(p); w, hgt = im.size; a = np.asarray(im.convert("L"), dtype=np.float32)
    b = max(2, int(0.08 * min(a.shape))); frame = np.concatenate([a[:b].ravel(), a[-b:].ravel(), a[:, :b].ravel(), a[:, -b:].ravel()])
    c = 16; corners = [a[:c, :c].mean(), a[:c, -c:].mean(), a[-c:, :c].mean(), a[-c:, -c:].mean()]
    return [w, hgt, w / hgt, os.path.getsize(p), float(im.mode == "L"), frame.mean(), frame.std(), *corners, a.mean(), a.std(), (a < 10).mean(), (a > 245).mean()]
Xe = np.array([feats(p) for p in ext_paths])
meta = pd.DataFrame({"class": [NAMES[v] for v in ext_y], "size": [f"{int(a)}x{int(b)}" for a, b in Xe[:, :2]], "kb": Xe[:, 3] / 1024, "gray": Xe[:, 4], "frame": Xe[:, 5]})
MS = meta.groupby("class").agg(images=("size", "size"), distinct_sizes=("size", "nunique"), most_common_size=("size", lambda s: s.value_counts().index[0]),
                               share_most_common=("size", lambda s: round(s.value_counts(normalize=True).iloc[0], 3)),
                               median_file_kb=("kb", lambda s: round(s.median(), 1)), grayscale_share=("gray", "mean"), median_frame_mean=("frame", lambda s: round(s.median(), 1)))
MS.to_csv(f"{OUT}/table_source_metadata.csv"); print("\n" + MS.to_string())
tr, te = ext_part != "test", ext_part == "test"
sc_in = HistGradientBoostingClassifier(random_state=0).fit(Xe[tr], (ext_y[tr] == NT).astype(int)).predict_proba(Xe[te])[:, 1]
sp_ = json.load(open(f"{W}/split_patient.json")); trp = sp_["train_paths"] + sp_["val_paths"]; try_ = np.array(sp_["train_y"] + sp_["val_y"])
sc_x = HistGradientBoostingClassifier(random_state=0).fit(np.array([feats(p) for p in trp]), (try_ == NT).astype(int)).predict_proba(Xe)[:, 1]
SCt = pd.DataFrame([{"Background-only classifier": "trained on BDNeuro train+val, tested on BDNeuro test", "No tumor vs tumor AUC": round(roc_auc_score(ext_y[te] == NT, sc_in), 4),
                     "Balanced accuracy": round(balanced_accuracy_score(ext_y[te] == NT, sc_in > .5), 4)},
                    {"Background-only classifier": "trained on Sartaj (patient split), tested on all BDNeuro", "No tumor vs tumor AUC": round(roc_auc_score(ext_y == NT, sc_x), 4),
                     "Balanced accuracy": round(balanced_accuracy_score(ext_y == NT, sc_x > .5), 4)}])
SCt.to_csv(f"{OUT}/table_background_only.csv", index=False); print("\n" + SCt.to_string(index=False))

# ---------- 3. external test: patient-split models, all four seeds, no retraining ----------
dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
keep = np.where(~overlap)[0]; kp = [ext_paths[i] for i in keep]; ky = ext_y[keep]
X = torch.stack([test_transform(Image.open(p).convert("RGB")) for p in kp])           # preprocessed once, reused by every model
def run(model):
    model.to(dev).eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), 64): out.append(F.softmax(model(X[i:i+64].to(dev)), 1).cpu())
    model.cpu(); return torch.cat(out).numpy()
def load(build, f):
    m = build(pretrained=False); m.load_state_dict(torch.load(f, map_location="cpu")); return m
P = {}
for s in [42, 43, 44, 45]:
    P[s] = {}
    for key, nm, b, d in [("resnet", "ResNet101", OptimizedResNet101, "runs_patient"), ("swin", "Swin-V2-B", make_swin, "runs_patient"),
                          ("effb0", "EfficientNet-B0", make_effb0, "runs_patient_baselines"), ("convnext", "ConvNeXt-T", make_convnext, "runs_patient_baselines"),
                          ("resnet50", "ResNet50", make_resnet50, "runs_patient_baselines")]:
        f = f"{W}/{d}/{key}_seed{s}.pt"
        if os.path.exists(f): P[s][nm] = run(load(b, f))
    if "ResNet101" in P[s] and "Swin-V2-B" in P[s]: P[s]["Equal fusion"] = (P[s]["ResNet101"] + P[s]["Swin-V2-B"]) / 2
    print(f"seed {s}: evaluated", list(P[s]))
np.savez(f"{OUT}/external_probs.npz", y=ky, paths=np.array(kp), **{f"{nm}_seed{s}": p for s in P for nm, p in P[s].items()})
def wilson(k, n, z=1.96):
    p = k/n; d = 1 + z*z/n; c = (p + z*z/(2*n))/d; h = z*sqrt(p*(1-p)/n + z*z/(4*n*n))/d; return c - h, c + h
models = [m for m in ["Equal fusion", "Swin-V2-B", "ResNet101", "EfficientNet-B0", "ConvNeXt-T", "ResNet50"] if all(m in P[s] for s in P)]
rows, n = [], len(ky)
for nm in models:
    acc = [100 * (P[s][nm].argmax(1) == ky).mean() for s in P]
    rec = [[100 * (P[s][nm].argmax(1)[ky == c] == c).mean() for s in P] for c in range(4)]
    sens = [100 * ((P[s][nm].argmax(1) != NT) & (ky != NT)).sum() / (ky != NT).sum() for s in P]
    spec = [100 * ((P[s][nm].argmax(1) == NT) & (ky == NT)).sum() / (ky == NT).sum() for s in P]
    lo, hi = wilson(int(round(np.mean(acc) / 100 * n)), n)
    rows.append({"Model": nm, "Accuracy % mean ± SD": f"{np.mean(acc):.2f} ± {np.std(acc, ddof=1):.2f}", "95% CI (images)": f"{100*lo:.1f} to {100*hi:.1f}",
                 **{f"{NAMES[c]} recall %": round(np.mean(rec[c]), 2) for c in range(4)},
                 "Tumor sensitivity %": round(np.mean(sens), 2), "No-tumor specificity %": round(np.mean(spec), 2), "Per-seed accuracy": [round(a, 2) for a in acc]})
EX = pd.DataFrame(rows); EX.to_csv(f"{OUT}/table_external_results.csv", index=False); print(f"\nExternal test on {n} BDNeuro-MRI images:\n" + EX.to_string(index=False))
comp = []
for other in [m for m in models if m != "Equal fusion"]:
    for s in P:
        a, b = P[s]["Equal fusion"].argmax(1) == ky, P[s][other].argmax(1) == ky; n10, n01 = int((a & ~b).sum()), int((~a & b).sum())
        comp.append({"Comparison": f"Equal fusion vs {other}", "Seed": s, "Difference (points)": round(100 * (a.mean() - b.mean()), 2),
                     "Only fusion correct": n10, "Only other correct": n01, "Exact McNemar p": round(binomtest(min(n10, n01), n10 + n01, .5).pvalue, 4) if n10 + n01 else 1.0})
CP = pd.DataFrame(comp); CP.to_csv(f"{OUT}/table_external_paired.csv", index=False); print("\n" + CP.to_string(index=False))
cm = confusion_matrix(ky, P[42]["Equal fusion"].argmax(1), labels=range(4))
print("\nConfusion matrix, equal fusion, seed 42 (rows = true):\n", pd.DataFrame(cm, index=NAMES, columns=NAMES))
pd.DataFrame(cm, index=NAMES, columns=NAMES).to_csv(f"{OUT}/confusion_matrix_fusion_seed42.csv")
zp = f"{W}/external_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*"): z.write(f, os.path.basename(f))
print(f"\nSaved {zp}")
try:
    from IPython.display import FileLink, display; os.chdir(W); display(FileLink("external_results.zip"))
except Exception: pass

In [ ]:
# ===== External dataset (PMRAM, original images only): overlap check, source check, external test =====
!pip install -q imagehash
import os, glob, json, zipfile, importlib, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, imagehash, torch, torch.nn.functional as F
from PIL import Image
from math import sqrt
from scipy.stats import binomtest
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix
W = "/kaggle/working"; OUT = f"{W}/external_pmram"; os.makedirs(OUT, exist_ok=True)
NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]; NT = 2
PMRAM_ROOT = "/kaggle/input/datasets/musmannasir/pmram-dataset"
RAW_DIR = None          # <- if the cell stops and asks, put the folder of ORIGINAL images here
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
def cls_of(name):
    n = re.sub(r"[^a-z]", "", name.lower())
    if "glioma" in n: return 0
    if "meningioma" in n: return 1
    if "pituitary" in n: return 3
    if "notumor" in n or "normal" in n or "healthy" in n or n == "no": return 2
    return None

# ---------- 0. find PMRAM and keep only the original (non-augmented) images ----------
assert os.path.isdir(PMRAM_ROOT), f"{PMRAM_ROOT} not found: attach the PMRAM dataset with Add Input, then rerun."
roots = [PMRAM_ROOT.rstrip("/") + "/"]
for z in glob.glob(os.path.join(PMRAM_ROOT, "**/*.zip"), recursive=True):               # zips inside the upload, if any
    d_ = os.path.join(W, "pmram_extracted", os.path.splitext(os.path.basename(z))[0])
    if not os.path.isdir(d_):
        with zipfile.ZipFile(z) as zf: zf.extractall(d_)
    roots.append(d_ + "/")
print("Folder tree (folders that contain images, with image counts):")
img_dirs = {}
for r in roots:
    for d_, _, files in os.walk(r):
        k = sum(f.lower().endswith(IMG_EXT) for f in files)
        if k: img_dirs[d_] = k; print(f"  {k:5d}  {d_}")
if RAW_DIR is None:
    cand = sorted({d_.rsplit("/", 1)[0] for d_ in img_dirs if re.search(r"raw|original", d_, re.I)}, key=len)
    if cand: RAW_DIR = cand[0]
assert RAW_DIR, ("Could not tell original from augmented images. Look at the tree above and set RAW_DIR at the top of this cell "
                 "to the folder that holds the ORIGINAL images (about 400 per class), then rerun.")
ext_paths, ext_y = [], []
for d_, k in img_dirs.items():
    if not d_.startswith(RAW_DIR.rstrip("/")): continue
    c = cls_of(os.path.basename(d_.rstrip("/")))
    if c is None: print("  skipping folder with unknown class:", d_); continue
    for f in sorted(glob.glob(os.path.join(d_, "*"))):
        if f.lower().endswith(IMG_EXT): ext_paths.append(f); ext_y.append(c)
ext_y = np.array(ext_y)
print(f"\nUsing ORIGINAL images under {RAW_DIR}: {len(ext_paths)} |", dict(zip(NAMES, np.bincount(ext_y, minlength=4).tolist())))
assert 1000 <= len(ext_paths) <= 2500, "Expected about 1,600 original images; check RAW_DIR (augmented images must not be used)."

# ---------- 1. overlap with Sartaj, Figshare, and duplicates inside PMRAM ----------
assert os.path.exists(f"{W}/train_seed.py"), "train_seed.py missing: run the %%writefile train_seed.py cell first."
import train_seed; importlib.reload(train_seed)
from train_seed import all_images, OptimizedResNet101, make_swin, make_convnext, make_resnet50, make_effb0, test_transform
ph = lambda p: int(str(imagehash.phash(Image.open(p).convert("L"))), 16)
s_paths, s_lab = all_images()
h_s = np.load(f"{W}/phash_all.npy") if os.path.exists(f"{W}/phash_all.npy") else np.array([ph(p) for p in s_paths], dtype=np.uint64)
if os.path.exists(f"{W}/figshare/mask_index.csv"): h_f = np.array([int(x, 16) for x in pd.read_csv(f"{W}/figshare/mask_index.csv")["hash"]], dtype=np.uint64)
else:
    fp = sorted(glob.glob(f"{W}/figshare/jpg/*.png")); h_f = np.array([ph(p) for p in fp], dtype=np.uint64)
    if not len(fp): print("WARNING: no Figshare hashes found; run the Figshare cell for the full overlap check.")
h_e = np.array([ph(p) for p in ext_paths], dtype=np.uint64)
def dist_to(a, b):
    if len(b) == 0: return np.full(len(a), 64), np.zeros(len(a), int)
    dm, im = np.empty(len(a), int), np.empty(len(a), int)
    for i in range(0, len(a), 256):
        d = np.unpackbits((a[i:i+256, None] ^ b[None, :]).view(np.uint8).reshape(len(a[i:i+256]), len(b), 8), axis=2).sum(2)
        dm[i:i+256], im[i:i+256] = d.min(1), d.argmin(1)
    return dm, im
d_s, i_s = dist_to(h_e, h_s); d_f, _ = dist_to(h_e, h_f)
D_in = np.unpackbits((h_e[:, None] ^ h_e[None, :]).view(np.uint8).reshape(len(h_e), len(h_e), 8), axis=2).sum(2); np.fill_diagonal(D_in, 64)
rows = []
for c in range(4):
    m = ext_y == c
    rows.append({"Class": NAMES[c], "Images": int(m.sum()), "Exact match to Sartaj": int((m & (d_s == 0)).sum()), "Sartaj within 5 bits": int((m & (d_s <= 5)).sum()),
                 "Sartaj within 8 bits": int((m & (d_s <= 8)).sum()), "Figshare within 5 bits": int((m & (d_f <= 5)).sum()),
                 "Figshare within 8 bits": int((m & (d_f <= 8)).sum()), "Near-copy of another PMRAM image (<=5 bits)": int((m & (D_in.min(1) <= 5)).sum())})
OV = pd.DataFrame(rows); OV.loc[len(OV)] = ["Total"] + OV.iloc[:, 1:].sum().tolist()
OV.to_csv(f"{OUT}/table_overlap.csv", index=False); print("\n" + OV.to_string(index=False))
overlap = (d_s <= 5) | (d_f <= 5)
lab_of = {"glioma_tumor": 0, "meningioma_tumor": 1, "no_tumor": 2, "pituitary_tumor": 3}
ex = d_s == 0
if ex.any():
    agree = np.array([lab_of.get(s_paths[j].split("/")[-2], -1) for j in i_s[ex]]) == ext_y[ex]
    print(f"Exact copies of Sartaj images: {int(ex.sum())}, same label in {int(agree.sum())}")
pd.DataFrame({"pmram_image": ext_paths, "label": [NAMES[v] for v in ext_y], "nearest_sartaj": [s_paths[j] for j in i_s],
              "dist_sartaj": d_s, "dist_figshare": d_f, "dist_nearest_pmram": D_in.min(1)}).to_csv(f"{OUT}/overlap_per_image.csv", index=False)
show = np.argsort(d_s)[:6]
fig, ax = plt.subplots(len(show), 2, figsize=(6, 3 * len(show)), squeeze=False)
for r, i in enumerate(show):
    for c_, (p, t) in enumerate([(ext_paths[i], f"PMRAM: {NAMES[ext_y[i]]}"), (s_paths[i_s[i]], f"Sartaj: {NAMES[s_lab[i_s[i]]]}")]):
        ax[r, c_].imshow(Image.open(p).convert("L"), cmap="gray"); ax[r, c_].axis("off"); ax[r, c_].set_title(f"{t} | {d_s[i]} bits", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/Fig_closest_sartaj_matches.png", dpi=150); plt.show()
print(f"\nImages kept for the external test (no Sartaj or Figshare image within 5 bits): {int((~overlap).sum())} of {len(ext_paths)}")
print(">>> If many images overlap, PMRAM is not independent: stop here and report it. <<<")

# ---------- 2. source check (PMRAM was resized to 512 x 512, so size cannot act as a cue; background still can) ----------
def feats(p):
    im = Image.open(p); w, hgt = im.size; a = np.asarray(im.convert("L"), dtype=np.float32)
    b = max(2, int(0.08 * min(a.shape))); frame = np.concatenate([a[:b].ravel(), a[-b:].ravel(), a[:, :b].ravel(), a[:, -b:].ravel()])
    c = 16; corners = [a[:c, :c].mean(), a[:c, -c:].mean(), a[-c:, :c].mean(), a[-c:, -c:].mean()]
    return [w, hgt, w / hgt, os.path.getsize(p), float(im.mode == "L"), frame.mean(), frame.std(), *corners, a.mean(), a.std(), (a < 10).mean(), (a > 245).mean()]
Xe = np.array([feats(p) for p in ext_paths])
meta = pd.DataFrame({"class": [NAMES[v] for v in ext_y], "size": [f"{int(a)}x{int(b)}" for a, b in Xe[:, :2]], "kb": Xe[:, 3] / 1024, "gray": Xe[:, 4], "frame": Xe[:, 5]})
MS = meta.groupby("class").agg(images=("size", "size"), distinct_sizes=("size", "nunique"), most_common_size=("size", lambda s: s.value_counts().index[0]),
                               median_file_kb=("kb", lambda s: round(s.median(), 1)), grayscale_share=("gray", "mean"), median_frame_mean=("frame", lambda s: round(s.median(), 1)))
MS.to_csv(f"{OUT}/table_source_metadata.csv"); print("\n" + MS.to_string())
cv = cross_val_predict(HistGradientBoostingClassifier(random_state=0), Xe, (ext_y == NT).astype(int), cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
sp_ = json.load(open(f"{W}/split_patient.json")); trp = sp_["train_paths"] + sp_["val_paths"]; try_ = np.array(sp_["train_y"] + sp_["val_y"])
sx = HistGradientBoostingClassifier(random_state=0).fit(np.array([feats(p) for p in trp]), (try_ == NT).astype(int)).predict_proba(Xe)[:, 1]
SCt = pd.DataFrame([{"Background-only classifier": "within PMRAM, 5-fold cross-validation", "No tumor vs tumor AUC": round(roc_auc_score(ext_y == NT, cv), 4),
                     "Balanced accuracy": round(balanced_accuracy_score(ext_y == NT, cv > .5), 4)},
                    {"Background-only classifier": "trained on Sartaj (patient split), tested on PMRAM", "No tumor vs tumor AUC": round(roc_auc_score(ext_y == NT, sx), 4),
                     "Balanced accuracy": round(balanced_accuracy_score(ext_y == NT, sx > .5), 4)}])
SCt.to_csv(f"{OUT}/table_background_only.csv", index=False); print("\n" + SCt.to_string(index=False))

# ---------- 3. external test on the kept original images: patient-split models, four seeds, no retraining ----------
dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
keep = np.where(~overlap)[0]; kp = [ext_paths[i] for i in keep]; ky = ext_y[keep]
X = torch.stack([test_transform(Image.open(p).convert("RGB")) for p in kp])
def run(model):
    model.to(dev).eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), 64): out.append(F.softmax(model(X[i:i+64].to(dev)), 1).cpu())
    model.cpu(); return torch.cat(out).numpy()
def load(build, f):
    m = build(pretrained=False); m.load_state_dict(torch.load(f, map_location="cpu")); return m
P = {}
for s in [42, 43, 44, 45]:
    P[s] = {}
    for key, nm, b, d in [("resnet", "ResNet101", OptimizedResNet101, "runs_patient"), ("swin", "Swin-V2-B", make_swin, "runs_patient"),
                          ("effb0", "EfficientNet-B0", make_effb0, "runs_patient_baselines"), ("convnext", "ConvNeXt-T", make_convnext, "runs_patient_baselines"),
                          ("resnet50", "ResNet50", make_resnet50, "runs_patient_baselines")]:
        f = f"{W}/{d}/{key}_seed{s}.pt"
        if os.path.exists(f): P[s][nm] = run(load(b, f))
    if "ResNet101" in P[s] and "Swin-V2-B" in P[s]: P[s]["Equal fusion"] = (P[s]["ResNet101"] + P[s]["Swin-V2-B"]) / 2
    print(f"seed {s}: evaluated", list(P[s]))
np.savez(f"{OUT}/external_probs.npz", y=ky, paths=np.array(kp), **{f"{nm}_seed{s}": p for s in P for nm, p in P[s].items()})
def wilson(k, n, z=1.96):
    p = k/n; d = 1 + z*z/n; c = (p + z*z/(2*n))/d; h = z*sqrt(p*(1-p)/n + z*z/(4*n*n))/d; return c - h, c + h
models = [m for m in ["Equal fusion", "Swin-V2-B", "ResNet101", "EfficientNet-B0", "ConvNeXt-T", "ResNet50"] if all(m in P[s] for s in P)]
rows, n = [], len(ky)
for nm in models:
    acc = [100 * (P[s][nm].argmax(1) == ky).mean() for s in P]
    rec = [[100 * (P[s][nm].argmax(1)[ky == c] == c).mean() for s in P] for c in range(4)]
    sens = [100 * ((P[s][nm].argmax(1) != NT) & (ky != NT)).sum() / (ky != NT).sum() for s in P]
    spec = [100 * ((P[s][nm].argmax(1) == NT) & (ky == NT)).sum() / (ky == NT).sum() for s in P]
    lo, hi = wilson(int(round(np.mean(acc) / 100 * n)), n)
    rows.append({"Model": nm, "Accuracy % mean ± SD": f"{np.mean(acc):.2f} ± {np.std(acc, ddof=1):.2f}", "95% CI (images)": f"{100*lo:.1f} to {100*hi:.1f}",
                 **{f"{NAMES[c]} recall %": round(np.mean(rec[c]), 2) for c in range(4)},
                 "Tumor sensitivity %": round(np.mean(sens), 2), "No-tumor specificity %": round(np.mean(spec), 2), "Per-seed accuracy": [round(float(a), 2) for a in acc]})
EX = pd.DataFrame(rows); EX.to_csv(f"{OUT}/table_external_results.csv", index=False); print(f"\nExternal test on {n} PMRAM original images:\n" + EX.to_string(index=False))
comp = []
for other in [m for m in models if m != "Equal fusion"]:
    for s in P:
        a, b = P[s]["Equal fusion"].argmax(1) == ky, P[s][other].argmax(1) == ky; n10, n01 = int((a & ~b).sum()), int((~a & b).sum())
        comp.append({"Comparison": f"Equal fusion vs {other}", "Seed": s, "Difference (points)": round(100 * (a.mean() - b.mean()), 2),
                     "Only fusion correct": n10, "Only other correct": n01, "Exact McNemar p": round(binomtest(min(n10, n01), n10 + n01, .5).pvalue, 4) if n10 + n01 else 1.0})
pd.DataFrame(comp).to_csv(f"{OUT}/table_external_paired.csv", index=False); print("\n" + pd.DataFrame(comp).to_string(index=False))
cm = confusion_matrix(ky, P[42]["Equal fusion"].argmax(1), labels=range(4))
print("\nConfusion matrix, equal fusion, seed 42 (rows = true):\n", pd.DataFrame(cm, index=NAMES, columns=NAMES))
pd.DataFrame(cm, index=NAMES, columns=NAMES).to_csv(f"{OUT}/confusion_matrix_fusion_seed42.csv")
zp = f"{W}/external_pmram_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*"): z.write(f, os.path.basename(f))
print(f"\nSaved {zp}")
try:
    from IPython.display import FileLink, display; os.chdir(W); display(FileLink("external_pmram_results.zip"))
except Exception: pass

## Pixel-level verification of hash matches
Checks the image(48) group, every Sartaj-to-Figshare patient-ID match (pixel correlation, ties between patients) and the matches between the candidate external datasets and Sartaj (byte identity, decoded-pixel identity, correlation).

In [ ]:
# ===== Pixel-level verification of hash matches (compliance audit): no GPU, no retraining =====
!pip install -q imagehash h5py
import os, glob, json, hashlib, importlib, re, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt, imagehash
from PIL import Image
W = "/kaggle/working"; OUT = f"{W}/verification"; os.makedirs(OUT, exist_ok=True)
NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
tail = lambda p: "/".join(str(p).split("/")[-3:])
import train_seed; importlib.reload(train_seed); from train_seed import all_images
s_paths, s_lab = all_images(); s_lab = np.array(s_lab)
def gray(p, size=256):
    return np.asarray(Image.open(p).convert("L").resize((size, size), Image.BILINEAR), dtype=np.float32)
def md5(p): return hashlib.md5(open(p, "rb").read()).hexdigest()
def compare(a, b):
    """byte identity, decoded-pixel identity (same size), and correlation after resizing to 256 x 256"""
    ia, ib = Image.open(a).convert("L"), Image.open(b).convert("L")
    same_px = ia.size == ib.size and np.array_equal(np.asarray(ia), np.asarray(ib))
    x, y = gray(a).ravel(), gray(b).ravel()
    r = float(np.corrcoef(x, y)[0, 1]) if x.std() > 0 and y.std() > 0 else 0.0
    return {"byte_identical": md5(a) == md5(b), "pixel_identical": bool(same_px), "pixel_corr": round(r, 4),
            "mean_abs_diff": round(float(np.abs(x - y).mean()), 2)}
ph = lambda p: int(str(imagehash.phash(Image.open(p).convert("L"))), 16)
hs = np.load(f"{W}/phash_all.npy") if os.path.exists(f"{W}/phash_all.npy") else np.array([ph(p) for p in s_paths], dtype=np.uint64)
def hd(a, b): return bin(int(a) ^ int(b)).count("1")
VERIFIED = 0.98                                   # pixel correlation treated as the same image

# ---------- 1. the image(48) group ----------
g = [i for i, p in enumerate(s_paths) if tail(p) in {"Testing/glioma_tumor/image(48).jpg"}]
assert g, "image(48) not found"
k48 = g[0]; mem = [j for j in range(len(s_paths)) if hd(hs[j], hs[k48]) <= 5 and j != k48]
rows = [{"image_a": tail(s_paths[k48]), "label_a": NAMES[s_lab[k48]], "image_b": tail(s_paths[j]), "label_b": NAMES[s_lab[j]],
         "hash_bits": hd(hs[k48], hs[j]), **compare(s_paths[k48], s_paths[j])} for j in mem]
T1 = pd.DataFrame(rows); T1.to_csv(f"{OUT}/image48_group.csv", index=False); print("image(48) and its hash neighbours:\n" + T1.to_string(index=False))
fig, ax = plt.subplots(1, min(4, len(mem)) + 1, figsize=(3 * (min(4, len(mem)) + 1), 3.3))
for a, j in zip(ax, [k48] + mem[:4]):
    a.imshow(Image.open(s_paths[j]).convert("L"), cmap="gray"); a.axis("off"); a.set_title(f"{NAMES[s_lab[j]]}\n{tail(s_paths[j])}", fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/Fig_image48_group.png", dpi=150); plt.show()

# ---------- 2. Sartaj -> Figshare patient-ID matches: pixel check, ties, healthy false matches ----------
fx = pd.read_csv(f"{W}/figshare/figshare_index.csv", dtype={"pid": str}) if os.path.exists(f"{W}/figshare/figshare_index.csv") else None
assert fx is not None and "path" in fx.columns, "figshare_index.csv not found: run the Figshare download cell first."
fx = fx[fx.path.map(os.path.exists)].reset_index(drop=True)
hf = np.array([ph(p) for p in fx.path], dtype=np.uint64)
rec = []
for i, p in enumerate(s_paths):
    d = np.unpackbits((np.array([hs[i]], dtype=np.uint64)[:, None] ^ hf[None, :]).view(np.uint8).reshape(1, len(hf), 8), axis=2).sum(2)[0]
    m = int(d.min())
    if m > 5: continue
    at_min = np.where(d == m)[0]; pids = set(fx.pid[at_min]); j = int(at_min[0])
    rec.append({"sartaj": tail(p), "sartaj_label": NAMES[s_lab[i]], "figshare": os.path.basename(fx.path[j]), "figshare_label": NAMES[int(fx.label[j])],
                "pid": fx.pid[j], "hash_bits": m, "ties_at_min": len(at_min), "tie_with_different_patient": len(pids) > 1,
                **{k: v for k, v in compare(p, fx.path[j]).items() if k in ("pixel_corr", "mean_abs_diff")}})
M = pd.DataFrame(rec); M["verified_same_image"] = M.pixel_corr >= VERIFIED; M.to_csv(f"{OUT}/patient_id_matches_verified.csv", index=False)
S = M.groupby("hash_bits").agg(matches=("sartaj", "size"), verified=("verified_same_image", "sum"), median_corr=("pixel_corr", "median"),
                               min_corr=("pixel_corr", "min"), ties_with_different_patient=("tie_with_different_patient", "sum"))
S.to_csv(f"{OUT}/patient_id_match_summary.csv"); print("\nPatient-ID matches by hash distance (pixel check):\n" + S.to_string())
H = M[M.sartaj_label == "No Tumor"]; print(f"\nHealthy Sartaj images matched to a Figshare tumor image: {len(H)}, verified as the same image: {int(H.verified_same_image.sum())}")
print(H[["sartaj", "figshare", "figshare_label", "hash_bits", "pixel_corr"]].to_string(index=False))
C = M[(M.sartaj_label != M.figshare_label) & (M.sartaj_label != "No Tumor")]
print(f"\nTumor label conflicts with Figshare: {len(C)}, verified as the same image: {int(C.verified_same_image.sum())}")
print(C[["sartaj", "sartaj_label", "figshare_label", "hash_bits", "pixel_corr"]].to_string(index=False))
U = M[~M.verified_same_image]
if len(U):
    k = min(6, len(U)); fig, ax = plt.subplots(k, 2, figsize=(6, 3 * k), squeeze=False)
    for r, (_, row) in enumerate(U.sort_values("pixel_corr").head(k).iterrows()):
        sp_ = s_paths[[tail(q) for q in s_paths].index(row.sartaj)]; fp_ = fx.path[list(fx.path.map(os.path.basename)).index(row.figshare)]
        for c, (q, t) in enumerate([(sp_, f"Sartaj {row.sartaj_label}"), (fp_, f"Figshare {row.figshare_label}")]):
            ax[r, c].imshow(Image.open(q).convert("L"), cmap="gray"); ax[r, c].axis("off"); ax[r, c].set_title(f"{t} | {row.hash_bits} bits | r = {row.pixel_corr}", fontsize=7)
    plt.tight_layout(); plt.savefig(f"{OUT}/Fig_unverified_patient_matches.png", dpi=150); plt.show()

# ---------- 3. external candidates: verify the 0-bit and near matches to Sartaj ----------
def find(pattern_dirs):
    for root in glob.glob("/kaggle/input/**/", recursive=True):
        if all(os.path.isdir(os.path.join(root, d)) for d in pattern_dirs): return root
    return None
cands = {}
bd = find(["train/glioma", "test/glioma"])
if bd: cands["BDNeuro-MRI"] = [(f, p) for p in glob.glob(os.path.join(bd, "*/*/*")) if p.lower().endswith((".jpg", ".jpeg", ".png")) for f in [None]]
pm = next((r for r in glob.glob("/kaggle/input/**/Raw Data/Raw/", recursive=True)), None)
if pm: cands["PMRAM"] = [(None, p) for p in glob.glob(os.path.join(pm, "*/*")) if p.lower().endswith((".jpg", ".jpeg", ".png"))]
ext = []
for name, items in cands.items():
    paths = [p for _, p in items]; he = np.array([ph(p) for p in paths], dtype=np.uint64)
    for i0 in range(0, len(paths), 256):
        d = np.unpackbits((he[i0:i0+256, None] ^ hs[None, :]).view(np.uint8).reshape(len(he[i0:i0+256]), len(hs), 8), axis=2).sum(2)
        for k, row in enumerate(d):
            m = int(row.min())
            if m <= 5:
                j = int(row.argmin()); p = paths[i0 + k]
                ext.append({"dataset": name, "image": tail(p), "sartaj": tail(s_paths[j]), "hash_bits": m, **compare(p, s_paths[j])})
    print(f"{name}: {len(paths)} images checked")
E = pd.DataFrame(ext)
if len(E):
    E["verified_same_image"] = E.pixel_corr >= VERIFIED; E.to_csv(f"{OUT}/external_matches_verified.csv", index=False)
    ES = E.groupby(["dataset", "hash_bits"]).agg(matches=("image", "size"), byte_identical=("byte_identical", "sum"), pixel_identical=("pixel_identical", "sum"),
                                                 verified_corr_098=("verified_same_image", "sum"), median_corr=("pixel_corr", "median"), min_corr=("pixel_corr", "min"))
    ES.to_csv(f"{OUT}/external_match_summary.csv"); print("\nExternal candidates vs Sartaj (pixel check):\n" + ES.to_string())
else: print("External datasets not attached: section 3 skipped (attach BDNeuro-MRI and PMRAM to include it).")

zp = f"{W}/verification_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*"): z.write(f, os.path.basename(f))
print(f"\nSaved {zp}")
try:
    from IPython.display import FileLink, display; os.chdir(W); display(FileLink("verification_results.zip"))
except Exception: pass